# BACI COUNTRY–PRODUCT BIPARTITE SLACK ANALYSIS

In [ ]:
# ============================================================
# BACI COUNTRY–PRODUCT BIPARTITE SLACK ANALYSIS - claude
# Purpose:
#   Build a country-product bipartite trade network from BACI
#   and compute quota-based weighted slack.
#
# Interpretation:
#   Country c is pivotal for product p if removing c leaves
#   the product market below a quota threshold.
#
# Author: Brian Daniel Bernhardt
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 0. Uploading libraries



In [37]:
# ============================================================
# 1. SET PATHS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Change this path to your BACI folder in Google Drive
DATA_DIR = "/content/drive/MyDrive/BACI_HS17_V202601_"

YEAR = 2023
THETA = 0.70   # With theta = 0.70, critical means supply share > 30%

baci_file = os.path.join(DATA_DIR, f"BACI_HS17_Y{YEAR}_V202601.csv")
country_file = os.path.join(DATA_DIR, "country_codes_V202601.csv")
product_file = os.path.join(DATA_DIR, "product_codes_HS17_V202601.csv")

print("BACI file:", baci_file)
print("Country file:", country_file)
print("Product file:", product_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BACI file: /content/drive/MyDrive/BACI_HS17_V202601_/BACI_HS17_Y2023_V202601.csv
Country file: /content/drive/MyDrive/BACI_HS17_V202601_/country_codes_V202601.csv
Product file: /content/drive/MyDrive/BACI_HS17_V202601_/product_codes_HS17_V202601.csv


### 1. Setting paths and parameters

This cell connects Google Colab to Google Drive and defines the location of the BACI data files used in the analysis. The empirical application is restricted to the year 2023. The parameter \(\theta = 0.70\) defines the quota threshold: a supplier is considered pivotal for a product when removing it leaves less than 70% of the product’s total observed supply. Equivalently, with \(\theta = 0.70\), a supplier becomes critical when its market share is greater than 30%.

In [ ]:
# ============================================================
# 2. LOAD LOOKUP TABLES
# ============================================================

country_codes = pd.read_csv(country_file)
product_codes = pd.read_csv(product_file, dtype={"code": str})

country_codes.columns = country_codes.columns.str.lower()
product_codes.columns = product_codes.columns.str.lower()

# Preserve HS6 product codes as 6-digit strings
product_codes["code"] = (
    product_codes["code"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

print("Country code columns:")
print(country_codes.columns.tolist())

print("\nProduct code columns:")
print(product_codes.columns.tolist())

country_codes.head(), product_codes.head()

### 2. Loading country and product lookup tables

This cell loads the auxiliary lookup tables used to interpret the BACI trade data. The country table links BACI country identifiers to country names and ISO codes, while the product table links HS product codes to their textual descriptions. These lookup tables are necessary because the raw BACI dataset contains numerical country and product identifiers rather than directly readable labels.

In [ ]:
# ============================================================
# 3. STANDARDIZE COUNTRY LOOKUP
# ============================================================

# BACI country files usually contain country_code and country_name.
# This block is defensive in case the exact column names differ.

possible_country_id_cols = ["country_code", "code", "i", "iso_3digit_alpha"]
possible_country_name_cols = ["country_name", "country", "name"]

country_id_col = None
country_name_col = None

for col in possible_country_id_cols:
    if col in country_codes.columns:
        country_id_col = col
        break

for col in possible_country_name_cols:
    if col in country_codes.columns:
        country_name_col = col
        break

if country_id_col is None or country_name_col is None:
    raise ValueError("Could not identify country code/name columns. Inspect country_codes.head().")

country_lookup = country_codes[[country_id_col, country_name_col]].copy()
country_lookup.columns = ["country_code", "country_name"]
country_lookup["country_code"] = country_lookup["country_code"].astype(int)

# Optional: keep ISO codes for maps or external matching
if "country_iso2" in country_codes.columns:
    country_lookup["country_iso2"] = country_codes["country_iso2"]

if "country_iso3" in country_codes.columns:
    country_lookup["country_iso3"] = country_codes["country_iso3"]

country_lookup.head()

country_lookup.head()

### 3. Standardizing the country lookup table

This cell prepares the country lookup table used to attach country names to the BACI trade flows. Since metadata files may use slightly different column names, the code first searches for the relevant country identifier and country-name columns. It then creates a standardized table with two columns: `country_code` and `country_name`. This lookup will later be merged with exporter and importer codes in the trade dataset.

In [ ]:
# ============================================================
# 4. LOAD BACI YEARLY FILE IN CHUNKS
# ============================================================

chunks = []
chunksize = 1_000_000

# For this analysis we only need:
# t = year, i = exporter, j = importer, k = product, v = trade value
usecols = ["t", "i", "j", "k", "v"]

for chunk in pd.read_csv(baci_file, usecols=usecols, chunksize=chunksize):

    # Ensure numerical trade values
    chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce")

    # Keep valid positive trade flows
    chunk = chunk.dropna(subset=["i", "k", "v"])
    chunk = chunk[chunk["v"] > 0].copy()

    # Standardize exporter and product codes
    chunk["i"] = chunk["i"].astype(int)
    chunk["k"] = chunk["k"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(6)

    # Aggregate HS6 products to HS4 and HS2
    chunk["hs4"] = chunk["k"].str[:4]
    chunk["hs2"] = chunk["k"].str[:2]

    # Aggregate inside each chunk: exporter country x HS4 product
    agg = (
        chunk
        .groupby(["i", "hs4", "hs2"], as_index=False)
        .agg(
            weight=("v", "sum"),
            n_records=("v", "size")
        )
    )

    chunks.append(agg)

country_product_raw = pd.concat(chunks, ignore_index=True)

# Aggregate again across chunks
country_product = (
    country_product_raw
    .groupby(["i", "hs4", "hs2"], as_index=False)
    .agg(
        weight=("weight", "sum"),
        n_records=("n_records", "sum")
    )
)

print("Country-HS4 edges:", country_product.shape)
country_product.head()

### 4. Loading and aggregating BACI trade flows

This cell loads the BACI yearly trade file in chunks to avoid memory problems. The original BACI data are reported at the exporter--importer--product level. For the global country--product analysis, the importer dimension is aggregated away, so each edge represents the total export value from country \(c\) to product \(p\), regardless of destination. Products are converted from HS6 to HS4 by keeping the first four digits of the HS6 code. The resulting dataset is a weighted bipartite network where exporter countries are connected to HS4 products, and edge weights represent total export value.

In [ ]:
# ============================================================
# 5. ADD COUNTRY NAMES
# ============================================================

country_product = country_product.merge(
    country_lookup,
    left_on="i",
    right_on="country_code",
    how="left"
)

country_product["country_name"] = country_product["country_name"].fillna(
    "C_" + country_product["i"].astype(str)
)

country_product = country_product.drop(columns=["country_code"])

country_product.head()

### 5. Adding country names to the exporter-product network

This cell merges the exporter-product dataset with the country lookup table. The raw BACI data identify exporter countries using numerical country codes, so this merge adds readable country names and ISO codes. If a country code is not found in the lookup table, the code assigns a fallback label using the original numerical identifier. This step makes the following tables and visualizations interpretable.

In [ ]:
# ============================================================
# 6a. OPTIONAL: ADD PRODUCT DESCRIPTIONS
# ============================================================

# Product code files can vary. We inspect likely columns.
product_codes.head()

### 6. Inspecting product descriptions

This cell inspects the BACI product lookup table. The lookup contains HS6 product codes and their corresponding textual descriptions. Since the main analysis is conducted at the HS4 level, the next step will aggregate HS6 descriptions into HS4-level product descriptions.

In [ ]:
# Try to detect product code and product description columns

possible_product_code_cols = ["product_code", "code", "k", "hs6"]
possible_product_desc_cols = ["description", "product_description", "name", "product"]

product_code_col = None
product_desc_col = None

for col in possible_product_code_cols:
    if col in product_codes.columns:
        product_code_col = col
        break

for col in possible_product_desc_cols:
    if col in product_codes.columns:
        product_desc_col = col
        break

if product_code_col is not None:
    product_lookup = product_codes.copy()

    # Safer conversion: reconstruct HS6 first, then truncate to HS4
    product_lookup[product_code_col] = (
        product_lookup[product_code_col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(6)
        .str[:4]
    )

    if product_desc_col is not None:
        product_lookup = product_lookup[[product_code_col, product_desc_col]].copy()
        product_lookup.columns = ["hs4", "product_description"]
    else:
        product_lookup = product_lookup[[product_code_col]].copy()
        product_lookup.columns = ["hs4"]
        product_lookup["product_description"] = product_lookup["hs4"]

    product_lookup = product_lookup.drop_duplicates("hs4")

    # Avoid duplicated description columns if the cell is rerun
    if "product_description" in country_product.columns:
        country_product = country_product.drop(columns=["product_description"])

    country_product = country_product.merge(product_lookup, on="hs4", how="left")

else:
    country_product["product_description"] = country_product["hs4"]

country_product["product_description"] = country_product["product_description"].fillna(
    country_product["hs4"]
)

country_product.head()

print("Unique HS4 codes in trade data:", country_product["hs4"].nunique())
print("Missing product descriptions:", country_product["product_description"].isna().sum())

country_product[["hs4", "product_description"]].drop_duplicates().head(20)

### 7. Adding product descriptions

This cell attaches readable product descriptions to the HS4 product codes used in the country-product network. Since the BACI product lookup is provided at the HS6 level, product codes are first standardized as six-digit strings and then truncated to HS4. The descriptions are used only for readability in tables and figures; the numerical analysis is based on HS4 codes.

In [ ]:
# ============================================================
# 7. COMPUTE COUNTRY–HS4 EDGE SLACK AND COUNTRY-LEVEL WEIGHTED SLACK
# ============================================================

# Interpretation:
#   c = exporter country
#   p = HS4 product
#
#   w_cp = exports of country c in product p
#   T_p  = total observed exports of product p in the thresholded network
#   Q_p  = theta * T_p
#
#   Slack_cp = (T_p - w_cp) / Q_p
#
# Critical condition:
#   Slack_cp < 1
#
# Equivalent:
#   w_cp / T_p > 1 - theta
#
# With theta = 0.70, a supplier is critical if it holds more than 30%
# of the observed product market.

THETA = 0.70

# Minimum trade value threshold.
# BACI values are usually in thousand USD.
# MIN_WEIGHT = 100 means at least 100,000 USD.
MIN_WEIGHT = 100

edges_cp = country_product.copy()

# Keep only economically meaningful exporter-product links
edges_cp = edges_cp[edges_cp["weight"] >= MIN_WEIGHT].copy()

edges_cp["year"] = YEAR

# Product total: T_p
edges_cp["T_p"] = edges_cp.groupby("hs4")["weight"].transform("sum")

# Product quota: Q_p = theta * T_p
edges_cp["Q_p"] = THETA * edges_cp["T_p"]

# Country share in each product market
edges_cp["supply_share_cp"] = edges_cp["weight"] / edges_cp["T_p"]

# Edge-level slack
edges_cp["Slack_cp"] = (edges_cp["T_p"] - edges_cp["weight"]) / edges_cp["Q_p"]

# Slack gap relative to the pivotality threshold
# Negative values indicate pivotal / critical links
edges_cp["slack_gap"] = edges_cp["Slack_cp"] - 1

# Criticality indicator
edges_cp["is_critical_supplier"] = edges_cp["Slack_cp"] < 1

# Depth of criticality:
# 0 if the supplier is not critical;
# positive if removing the supplier pushes the system below the quota.
edges_cp["criticality_depth"] = np.maximum(0, 1 - edges_cp["Slack_cp"])

# Criticality label
# With theta = 0.70, max Slack is 1/theta = 1.4286.
# Therefore, the near-critical band must be close to 1.

edges_cp["risk_label"] = np.select(
    [
        edges_cp["Slack_cp"] < 0.5,
        edges_cp["Slack_cp"] < 1.0,
        edges_cp["Slack_cp"] < 1.1,
        edges_cp["Slack_cp"] < 1.2
    ],
    [
        "SEVERE",
        "CRITICAL",
        "NEAR-CRITICAL",
        "WATCH"
    ],
    default="REDUNDANT"
)

# ============================================================
# 8. COUNTRY-LEVEL WEIGHTED SLACK
# ============================================================

# Country export portfolio:
#   s_c = sum_p w_cp
#
# We group by both country code and country name to avoid ambiguity.

edges_cp["s_c"] = edges_cp.groupby(["i", "country_name"])["weight"].transform("sum")

# Portfolio weight:
#   beta_cp = w_cp / s_c

edges_cp["beta_cp"] = edges_cp["weight"] / edges_cp["s_c"]

# Weighted slack contribution:
#   beta_cp * Slack_cp

edges_cp["country_weighted_slack_contribution"] = (
    edges_cp["beta_cp"] * edges_cp["Slack_cp"]
)

# ============================================================
# DIAGNOSTIC CHECKS
# ============================================================

print("========== NETWORK SIZE AFTER THRESHOLD ==========")
print("Minimum trade threshold:", MIN_WEIGHT)
print("Exporter-product edges:", edges_cp.shape[0])
print("Exporter countries:", edges_cp["country_name"].nunique())
print("HS4 products:", edges_cp["hs4"].nunique())

print("\n========== SLACK SUMMARY ==========")
print(edges_cp["Slack_cp"].describe())

print("\n========== CRITICAL EDGES ==========")
n_critical = edges_cp["is_critical_supplier"].sum()
share_critical = 100 * edges_cp["is_critical_supplier"].mean()

print("Critical edges:", n_critical)
print("Share of critical edges:", round(share_critical, 2), "%")

print("\nRisk label distribution:")
print(edges_cp["risk_label"].value_counts())

print("\nRisk label distribution (%):")
print(round(100 * edges_cp["risk_label"].value_counts(normalize=True), 2))

print("\n========== BETA CHECK ==========")
beta_check = (
    edges_cp
    .groupby(["i", "country_name"], as_index=False)
    .agg(beta_sum=("beta_cp", "sum"))
    .sort_values("beta_sum")
)

print("Lowest beta sums:")
print(beta_check.head())

print("\nHighest beta sums:")
print(beta_check.tail())

print("\nMaximum absolute deviation from 1:")
print(np.max(np.abs(beta_check["beta_sum"] - 1)))

print("\n========== SAMPLE OF COMPUTED EDGES ==========")
display(
    edges_cp[
        [
            "year",
            "country_name",
            "hs4",
            "product_description",
            "weight",
            "T_p",
            "supply_share_cp",
            "Slack_cp",
            "slack_gap",
            "criticality_depth",
            "risk_label",
            "beta_cp",
            "country_weighted_slack_contribution"
        ]
    ].head(10)
)

### 8. Computing link-level slack and country-level weighted slack

This cell computes the core slack metric for each exporter-country--HS4-product edge. For each product \(p\), total observed supply is defined as \(T_p=\sum_c w_{cp}\), and the feasibility quota is \(Q_p=\theta T_p\). Link-level slack is defined as \(Slack_{cp}=(T_p-w_{cp})/Q_p\). A supplier is pivotal when \(Slack_{cp}<1\), meaning that removing the supplier would leave less than the required quota of total product supply. With \(\theta=0.70\), this condition is equivalent to the supplier holding more than 30% of the observed product market.

The cell also computes country-level weighted slack. For each exporter \(c\), product-level slack values are weighted by the importance of each product in the exporter’s own export portfolio, \(\beta_{cp}=w_{cp}/\sum_p w_{cp}\). This creates a portfolio-level fragility measure: countries with low weighted slack are those whose economically important export products are structurally pivotal in global supply. The diagnostic checks verify the network size, the distribution of slack values, the number of critical edges, and whether the country-level portfolio weights correctly sum to one.

The diagnostic results show that the thresholded global exporter-product network contains 101,732 country-HS4 edges, covering 226 exporter countries and 1,222 HS4 products. Only 469 edges, corresponding to 0.46% of the network, satisfy the criticality condition \(Slack_{cp}<1\). With \(\theta=0.70\), this means that only a small subset of suppliers accounts for more than 30% of the observed supply of a given HS4 product. The slack criterion is therefore highly selective: it does not simply identify large exporters, but rather supplier-product links whose removal would push the product market below the predefined feasibility quota.

The slack distribution is concentrated close to its theoretical upper bound, \(1/\theta=1.4286\), because most countries hold very small shares in most HS4 product markets. This implies that risk labels must be defined relative to the bounded scale of the slack metric. In particular, a broad near-critical threshold such as \(Slack_{cp}<1.5\) is not informative when \(\theta=0.70\), since almost all non-critical links necessarily fall below this value. For this reason, subsequent classifications use narrower bands around the pivotality threshold \(Slack_{cp}=1\).

In [ ]:

# ============================================================
# 9. COUNTRY-LEVEL SUMMARY
# ============================================================

# This table aggregates link-level slack measures at the exporter-country level.
# It separates three dimensions:
#   1. Portfolio pivotality intensity: Slack_weighted
#   2. Systemic breadth: n_critical_products
#   3. Economic exposure: critical_export_value

country_slack = (
    edges_cp
    .groupby(["year", "i", "country_name"], as_index=False)
    .agg(
        total_exports=("weight", "sum"),
        n_products=("hs4", "nunique"),
        n_critical_products=("is_critical_supplier", "sum"),
        n_severe_products=("Slack_cp", lambda x: (x < 0.5).sum()),
        Slack_mean=("Slack_cp", "mean"),
        Slack_min=("Slack_cp", "min"),
        Slack_weighted=("country_weighted_slack_contribution", "sum"),
        max_supply_share=("supply_share_cp", "max")
    )
)

# Share of exported HS4 products that are critical
country_slack["critical_product_share"] = (
    country_slack["n_critical_products"] / country_slack["n_products"]
)

# ============================================================
# ADD ECONOMIC EXPOSURE AND CRITICALITY DEPTH
# ============================================================

country_extra = (
    edges_cp
    .assign(
        critical_export_value=lambda df: np.where(
            df["Slack_cp"] < 1,
            df["weight"],
            0
        ),
        severe_export_value=lambda df: np.where(
            df["Slack_cp"] < 0.5,
            df["weight"],
            0
        ),
        weighted_criticality_depth=lambda df: (
            df["beta_cp"] * df["criticality_depth"]
        )
    )
    .groupby(["year", "i", "country_name"], as_index=False)
    .agg(
        critical_export_value=("critical_export_value", "sum"),
        severe_export_value=("severe_export_value", "sum"),
        weighted_criticality_depth=("weighted_criticality_depth", "sum")
    )
)

country_slack = country_slack.merge(
    country_extra,
    on=["year", "i", "country_name"],
    how="left"
)

# Share of each country's exports that belongs to critical product links
country_slack["critical_export_share"] = (
    country_slack["critical_export_value"] / country_slack["total_exports"]
)

country_slack["severe_export_share"] = (
    country_slack["severe_export_value"] / country_slack["total_exports"]
)

# ============================================================
# RANKING VARIABLES
# ============================================================

# Lower Slack_weighted = higher portfolio pivotality intensity
country_slack["rank_weighted_slack"] = country_slack["Slack_weighted"].rank(
    method="min",
    ascending=True
)

# More critical products = broader systemic relevance
country_slack["rank_critical_products"] = country_slack["n_critical_products"].rank(
    method="min",
    ascending=False
)

# Larger critical export value = higher economic exposure
country_slack["rank_critical_export_value"] = country_slack["critical_export_value"].rank(
    method="min",
    ascending=False
)

# Combined descriptive ranking
country_slack["combined_country_rank"] = (
    country_slack["rank_weighted_slack"]
    + country_slack["rank_critical_products"]
    + country_slack["rank_critical_export_value"]
)

# Main display: countries with strong portfolio pivotality,
# broad critical-product coverage, and relevant critical export value.
country_slack = country_slack.sort_values(
    [
        "combined_country_rank",
        "Slack_weighted",
        "n_critical_products",
        "critical_export_value"
    ],
    ascending=[True, True, False, False]
)

display(
    country_slack[
        [
            "year",
            "country_name",
            "total_exports",
            "n_products",
            "n_critical_products",
            "n_severe_products",
            "critical_product_share",
            "critical_export_value",
            "critical_export_share",
            "severe_export_value",
            "severe_export_share",
            "Slack_mean",
            "Slack_min",
            "Slack_weighted",
            "weighted_criticality_depth",
            "max_supply_share",
            "rank_weighted_slack",
            "rank_critical_products",
            "rank_critical_export_value",
            "combined_country_rank"
        ]
    ].head(30)
)

# Alternative views useful for interpretation
print("\nTop countries by portfolio pivotality intensity:")
display(
    country_slack
    .sort_values("Slack_weighted", ascending=True)
    [
        [
            "country_name",
            "Slack_weighted",
            "n_critical_products",
            "critical_export_value",
            "critical_export_share",
            "total_exports"
        ]
    ]
    .head(15)
)

print("\nTop countries by systemic breadth:")
display(
    country_slack
    .sort_values("n_critical_products", ascending=False)
    [
        [
            "country_name",
            "n_critical_products",
            "n_severe_products",
            "critical_product_share",
            "Slack_weighted",
            "critical_export_value",
            "total_exports"
        ]
    ]
    .head(15)
)

print("\nTop countries by critical export value:")
display(
    country_slack
    .sort_values("critical_export_value", ascending=False)
    [
        [
            "country_name",
            "critical_export_value",
            "critical_export_share",
            "n_critical_products",
            "Slack_weighted",
            "total_exports"
        ]
    ]
    .head(15)
)

### 9. Country-level slack summary

This cell aggregates link-level slack measures at the exporter-country level. For each country, it reports total exports, the number of HS4 products exported, the number of critical and severe product links, average slack, minimum slack, and country-level weighted slack. The weighted slack measure summarizes the fragility of a country’s export portfolio by weighting each product-specific slack value according to the product’s importance in that country’s own exports. Lower values of weighted slack indicate that a country’s economically important export products are closer to, or below, the pivotality threshold.

This country-level aggregation should be interpreted carefully. A country may have very low weighted slack because one dominant product in its export basket is highly pivotal, even if the country is not broadly systemic across many products. Therefore, weighted slack captures portfolio-level pivotality intensity, while the number of critical products captures systemic breadth.

The country-level results distinguish three dimensions of exporter pivotality. First, `Slack_weighted` captures portfolio-level pivotality intensity: countries with lower weighted slack have export baskets in which economically important products are closer to, or below, the pivotality threshold. Second, `n_critical_products` captures systemic breadth, measuring how many HS4 products depend critically on a given exporter. Third, `critical_export_value` captures the economic scale of exports associated with critical supplier-product links.

The results show that China is the most relevant supplier across the combined ranking. It has 282 critical HS4 products, 25 severe products, and approximately 51.4% of its exports in the thresholded network are associated with critical supplier-product links. This indicates that China is not only important because of export volume, but because it is structurally pivotal across a very broad set of product markets.

Other countries appear relevant for different reasons. Australia ranks highly because almost half of its exports in the thresholded network are associated with critical product links, although its systemic breadth is much smaller than China’s. Indonesia and Brazil also combine relatively low weighted slack with substantial critical export values. The United States has a large number of critical products, but a relatively small share of its total exports is classified as critical, suggesting broad but less portfolio-concentrated pivotality.

The separate rankings also show why country-level slack should not be interpreted as a simple size ranking. Guinea has the lowest weighted slack, but this is driven by a narrow export portfolio in which one dominant product is highly pivotal. This makes Guinea important in terms of portfolio-level pivotality intensity, but not in terms of broad systemic relevance. For the paper, the key empirical message is that global supply pivotality has multiple dimensions: some countries are critical because they dominate specific products, while others are systemically important because they are pivotal across many product markets.

Note: The combined ranking is a descriptive synthesis of three dimensions: weighted slack, number of critical products, and critical export value. It should not be interpreted as a formal welfare ranking or as a direct geopolitical-risk index. Geopolitical vulnerability emerges when structural pivotality overlaps with political, logistical, or strategic risk.

In [ ]:
# ============================================================
# 10. PRODUCT-LEVEL THRESHOLD-SENSITIVE FRAGILITY
# ============================================================

# IMPORTANT:
# We do NOT use product-level weighted slack here because:
#
#   sum_c supply_share_cp * Slack_cp = (1 - HHI_p) / theta
#
# Therefore, product-level weighted slack is mechanically equivalent
# to an inverse transformation of HHI.
#
# Instead, we use a threshold-sensitive fragility score:
#
#   criticality_depth_cp = max(0, 1 - Slack_cp)
#
# This is positive only when the supplier is pivotal.
#
#   product_fragility_score_p = sum_c supply_share_cp * criticality_depth_cp
#
# This captures the intensity of product fragility conditional on
# crossing the pivotality threshold.

# Make sure criticality variables exist
edges_cp["is_critical_supplier"] = edges_cp["Slack_cp"] < 1

edges_cp["criticality_depth"] = np.maximum(
    0,
    1 - edges_cp["Slack_cp"]
)

# Share of product p supplied by critical suppliers
edges_cp["critical_supplier_share_contribution"] = (
    edges_cp["supply_share_cp"] *
    edges_cp["is_critical_supplier"].astype(int)
)

# Threshold-sensitive product fragility contribution
edges_cp["product_fragility_contribution"] = (
    edges_cp["supply_share_cp"] *
    edges_cp["criticality_depth"]
)

# Product-level aggregation
product_slack = (
    edges_cp
    .groupby(["year", "hs4", "hs2", "product_description"], as_index=False)
    .agg(
        total_trade=("weight", "sum"),
        n_exporters=("country_name", "nunique"),

        # Standard concentration benchmarks
        max_supply_share=("supply_share_cp", "max"),
        HHI=("supply_share_cp", lambda x: np.sum(np.square(x))),
        entropy=("supply_share_cp", lambda x: -np.sum(x * np.log(x + 1e-12))),

        # Slack-based summaries
        Slack_mean=("Slack_cp", "mean"),
        Slack_min=("Slack_cp", "min"),

        # Critical supplier structure
        n_critical_suppliers=("is_critical_supplier", "sum"),
        n_severe_suppliers=("Slack_cp", lambda x: (x < 0.5).sum()),
        pct_critical_suppliers=("is_critical_supplier", lambda x: 100 * x.mean()),

        # Threshold-sensitive product fragility
        critical_supplier_share=("critical_supplier_share_contribution", "sum"),
        product_fragility_score=("product_fragility_contribution", "sum"),
        max_criticality_depth=("criticality_depth", "max")
    )
)

# Effective number of suppliers based on HHI
product_slack["effective_n_exporters"] = 1 / product_slack["HHI"]

# Flag products with at least one pivotal supplier
product_slack["has_critical_supplier"] = (
    product_slack["n_critical_suppliers"] > 0
)

# ============================================================
# ADD DOMINANT SUPPLIER FOR EACH PRODUCT
# ============================================================

dominant_idx = edges_cp.groupby(["year", "hs4"])["supply_share_cp"].idxmax()

dominant_supplier = (
    edges_cp
    .loc[
        dominant_idx,
        [
            "year",
            "hs4",
            "country_name",
            "supply_share_cp",
            "Slack_cp",
            "risk_label"
        ]
    ]
    .rename(columns={
        "country_name": "dominant_supplier",
        "supply_share_cp": "dominant_supplier_share",
        "Slack_cp": "dominant_supplier_slack",
        "risk_label": "dominant_supplier_risk"
    })
)

product_slack = product_slack.merge(
    dominant_supplier,
    on=["year", "hs4"],
    how="left"
)

# ============================================================
# SORT PRODUCTS BY THRESHOLD-SENSITIVE FRAGILITY
# ============================================================

product_slack = product_slack.sort_values(
    [
        "product_fragility_score",
        "critical_supplier_share",
        "total_trade"
    ],
    ascending=[False, False, False]
)

# ============================================================
# DIAGNOSTICS
# ============================================================

print("========== PRODUCT-LEVEL SUMMARY ==========")
print("Number of HS4 products:", product_slack["hs4"].nunique())
print("Products with at least one critical supplier:", product_slack["has_critical_supplier"].sum())
print(
    "Share of products with at least one critical supplier:",
    round(100 * product_slack["has_critical_supplier"].mean(), 2),
    "%"
)

print("\n========== PRODUCT FRAGILITY SCORE SUMMARY ==========")
print(product_slack["product_fragility_score"].describe())

print("\n========== HHI SUMMARY ==========")
print(product_slack["HHI"].describe())

print("\n========== TOP FRAGILE PRODUCTS ==========")
display(
    product_slack[
        [
            "year",
            "hs4",
            "hs2",
            "product_description",
            "total_trade",
            "n_exporters",
            "effective_n_exporters",
            "dominant_supplier",
            "dominant_supplier_share",
            "dominant_supplier_slack",
            "dominant_supplier_risk",
            "HHI",
            "Slack_min",
            "n_critical_suppliers",
            "n_severe_suppliers",
            "critical_supplier_share",
            "product_fragility_score",
            "max_criticality_depth"
        ]
    ].head(30)
)

### 10. Product-level fragility
The product-level results show that 459 out of 1,222 HS4 products, corresponding to 37.56% of the analyzed product space, have at least one critical supplier. This means that for more than one third of HS4 products, at least one exporter holds a sufficiently large market share such that its removal would push the product market below the quota threshold defined by \(\theta=0.70\).

The `product_fragility_score` is highly skewed. Its median is zero, meaning that at least half of HS4 products have no pivotal supplier under the selected threshold. However, the upper tail contains products with very high fragility values. These are products where one dominant supplier accounts for a large share of observed supply and where the removal of that supplier would generate a large shortfall relative to the quota.

The top-ranked products are characterized by very high dominant-supplier shares and low effective numbers of exporters. For example, several products have one supplier accounting for more than 70% or 80% of observed supply, which produces severe slack values below 0.5. In these cases, the product market is not merely concentrated; it is structurally dependent on a specific supplier according to the quota-based pivotality rule.

HHI is retained as a benchmark concentration measure, but it is not used as the main fragility score. The key difference is that HHI summarizes concentration across all suppliers, while the slack-based fragility score activates only when a supplier crosses the pivotality threshold. Therefore, the product fragility score is threshold-sensitive: it measures whether concentration becomes structurally critical under the selected supply-feasibility rule.

The results should also be interpreted with caution. The highest fragility scores often correspond to relatively specialized or lower-value products, such as cocoa waste, jute yarn, artificial flowers, or specific textile and household products. These cases are useful for validating the metric because they reveal near-monopolistic supplier structures. However, for the paper’s policy argument on global imbalances, strategic autonomy, and supply-chain vulnerability, it will be necessary to complement this ranking with trade value, sectoral relevance, and Italy-specific import exposure.

In [ ]:
# ============================================================
# 11. CRITICAL COUNTRY–PRODUCT EDGES
# ============================================================

# Critical country-product edges are links where:
#   Slack_cp < 1
#
# Equivalently, with theta = 0.70:
#   supply_share_cp > 0.30
#
# These are supplier-product pairs where removing the supplier
# would push the product market below the quota threshold.

critical_edges = edges_cp[edges_cp["Slack_cp"] < 1].copy()

# Additional contribution measures for interpretation
critical_edges["country_portfolio_criticality_contribution"] = (
    critical_edges["beta_cp"] * critical_edges["criticality_depth"]
)

critical_edges["critical_export_value"] = critical_edges["weight"]

# Add product-level fragility information if product_slack already exists
product_fragility_cols = [
    "year",
    "hs4",
    "HHI",
    "effective_n_exporters",
    "n_exporters",
    "n_critical_suppliers",
    "critical_supplier_share",
    "product_fragility_score"
]

if "product_slack" in globals():
    critical_edges = critical_edges.merge(
        product_slack[product_fragility_cols],
        on=["year", "hs4"],
        how="left",
        suffixes=("", "_product")
    )

# ============================================================
# RANKING VARIABLES
# ============================================================

# More severe links have lower Slack_cp
critical_edges["rank_severity"] = critical_edges["Slack_cp"].rank(
    method="min",
    ascending=True
)

# Larger critical export values are economically more relevant
critical_edges["rank_trade_value"] = critical_edges["critical_export_value"].rank(
    method="min",
    ascending=False
)

# Larger product fragility score means the product market is more fragile
if "product_fragility_score" in critical_edges.columns:
    critical_edges["rank_product_fragility"] = critical_edges["product_fragility_score"].rank(
        method="min",
        ascending=False
    )
else:
    critical_edges["rank_product_fragility"] = np.nan

# Larger country-portfolio contribution means the critical link matters more
# inside the exporter country's own export basket.
critical_edges["rank_country_portfolio_impact"] = (
    critical_edges["country_portfolio_criticality_contribution"]
    .rank(method="min", ascending=False)
)

# Combined descriptive ranking
critical_edges["combined_critical_edge_rank"] = (
    critical_edges["rank_severity"]
    + critical_edges["rank_trade_value"]
    + critical_edges["rank_country_portfolio_impact"]
)

# ============================================================
# MAIN TABLE: COMBINED CRITICAL EDGE RANKING
# ============================================================

critical_edges_combined = critical_edges.sort_values(
    [
        "combined_critical_edge_rank",
        "Slack_cp",
        "critical_export_value"
    ],
    ascending=[True, True, False]
)

display_cols = [
    "year",
    "country_name",
    "hs4",
    "hs2",
    "product_description",
    "critical_export_value",
    "T_p",
    "supply_share_cp",
    "Slack_cp",
    "criticality_depth",
    "risk_label",
    "beta_cp",
    "country_portfolio_criticality_contribution"
]

optional_cols = [
    "HHI",
    "effective_n_exporters",
    "n_exporters",
    "n_critical_suppliers",
    "critical_supplier_share",
    "product_fragility_score",
    "rank_severity",
    "rank_trade_value",
    "rank_country_portfolio_impact",
    "combined_critical_edge_rank"
]

display_cols = display_cols + [col for col in optional_cols if col in critical_edges_combined.columns]

print("========== CRITICAL COUNTRY–PRODUCT EDGES ==========")
print("Number of critical edges:", critical_edges.shape[0])
print("Number of exporter countries with at least one critical edge:", critical_edges["country_name"].nunique())
print("Number of HS4 products with at least one critical edge:", critical_edges["hs4"].nunique())

print("\nRisk label distribution among critical edges:")
print(critical_edges["risk_label"].value_counts())

print("\nTop critical edges by combined ranking:")
display(critical_edges_combined[display_cols].head(30))

# ============================================================
# ALTERNATIVE VIEW 1: MOST SEVERE CRITICAL EDGES
# ============================================================

print("\nTop critical edges by severity:")
display(
    critical_edges
    .sort_values(["Slack_cp", "supply_share_cp"], ascending=[True, False])
    [display_cols]
    .head(20)
)

# ============================================================
# ALTERNATIVE VIEW 2: LARGEST CRITICAL EDGES BY TRADE VALUE
# ============================================================

print("\nTop critical edges by trade value:")
display(
    critical_edges
    .sort_values(["critical_export_value", "supply_share_cp"], ascending=[False, False])
    [display_cols]
    .head(20)
)

# ============================================================
# ALTERNATIVE VIEW 3: CRITICAL EDGES WITH HIGHEST COUNTRY-PORTFOLIO IMPACT
# ============================================================

print("\nTop critical edges by country-portfolio impact:")
display(
    critical_edges
    .sort_values(
        ["country_portfolio_criticality_contribution", "critical_export_value"],
        ascending=[False, False]
    )
    [display_cols]
    .head(20)
)

### 11. Critical country-product edges

This cell isolates the country-product links that satisfy the pivotality condition \(Slack_{cp}<1\). These are the supplier-product pairs for which removing the exporter would leave the product market below the quota threshold. With \(\theta=0.70\), this corresponds to suppliers holding more than 30% of the observed HS4 product market.

The cell reports critical edges from three complementary perspectives. The severity view ranks links by the lowest slack values, identifying near-monopoly or highly concentrated supplier-product structures. The trade-value view ranks critical links by export value, highlighting economically relevant dependencies. The country-portfolio-impact view weights criticality by the importance of the product inside the exporter’s own export basket. This distinction is important because some links are structurally severe but economically small, while others are less extreme but more relevant for global trade and supply-chain policy.

The critical-edge analysis identifies 469 country-product links satisfying \(Slack_{cp}<1\). These links involve 55 exporter countries and 459 HS4 products, showing that pivotality is relatively rare at the edge level but spread across a substantial part of the product space. Among the critical links, 40 are classified as severe and 429 as critical.

The three rankings provide complementary interpretations. The severity ranking highlights near-monopoly structures, where one country accounts for a very large share of observed supply. Examples include Côte d'Ivoire in cocoa waste, Bangladesh in jute yarn, China in artificial flowers, and Indonesia in lignite. These links have very low slack values and therefore represent cases of strong structural dependence.

The trade-value ranking highlights economically large critical links. Here, China appears prominently in products such as line telephone sets, automatic data processing machines, electric accumulators, electrical apparatus, and printed circuits. Australia also appears in iron ores and coal, while Brazil appears in soybeans. These links are particularly relevant for the paper because they connect the slack framework to global supply chains, strategic autonomy, and trade fragmentation.

The country-portfolio-impact ranking shows which critical links matter most within the exporter’s own trade basket. This view gives greater relevance to countries such as Guinea, Australia, Papua New Guinea, Zambia, Brazil, Madagascar, Indonesia, and the Democratic Republic of the Congo. These cases show that a supplier can be pivotal not only from the product-market perspective, but also because the critical product is central to the exporter’s own export structure.

Overall, the results confirm that criticality is multidimensional. Some links are severe because they are close to monopoly structures; others are important because of their trade value; and others matter because they dominate the exporter’s own portfolio. For the paper, the most policy-relevant results are those combining structural criticality with large trade values and strategic sectors.

In [ ]:
# ============================================================
# 12. TOP PIVOTAL COUNTRIES BY PORTFOLIO WEIGHTED SLACK
# ============================================================

# This ranking focuses only on portfolio-level pivotality intensity.
# Lower Slack_weighted means that the country's economically important
# export products are closer to, or below, the pivotality threshold.
#
# This is NOT a systemic importance ranking.
# For systemic relevance, use n_critical_products, critical_export_value,
# or combined_country_rank.

top_pivotal_countries_by_slack = (
    country_slack
    .query("n_critical_products > 0")
    .sort_values(
        ["Slack_weighted", "n_critical_products", "critical_export_value"],
        ascending=[True, False, False]
    )
)

display(
    top_pivotal_countries_by_slack[
        [
            "year",
            "country_name",
            "total_exports",
            "n_products",
            "n_critical_products",
            "n_severe_products",
            "critical_product_share",
            "critical_export_value",
            "critical_export_share",
            "Slack_mean",
            "Slack_weighted",
            "Slack_min",
            "max_supply_share",
            "weighted_criticality_depth"
        ]
    ].head(15)
)

### 12. Top pivotal countries by portfolio weighted slack

This cell ranks exporter countries by country-level weighted slack. This ranking captures portfolio-level pivotality intensity: countries with lower `Slack_weighted` have export baskets in which economically important products are closer to, or below, the pivotality threshold. This view is useful for identifying countries whose own export structure is strongly tied to pivotal products.

This ranking should not be interpreted as a general measure of systemic importance. A country may rank highly because one dominant export product is highly pivotal, even if it is not critical across many product markets. For this reason, the table should be read together with `n_critical_products` and `critical_export_value`, which capture systemic breadth and economic scale.

In [ ]:
# ============================================================
# 13. TOP FRAGILE PRODUCTS
# ============================================================

# This table identifies HS4 products with at least one pivotal supplier.
# Products are ranked by the threshold-sensitive product_fragility_score,
# not by product-level weighted slack.
#
# product_fragility_score = sum_c supply_share_cp * max(0, 1 - Slack_cp)
#
# HHI is retained only as a benchmark concentration measure.

top_fragile_products = (
    product_slack
    .query("n_critical_suppliers > 0")
    .sort_values(
        ["product_fragility_score", "critical_supplier_share", "total_trade"],
        ascending=[False, False, False]
    )
    .copy()
)

print("Number of products with at least one critical supplier:", top_fragile_products.shape[0])

display(
    top_fragile_products[
        [
            "year",
            "hs4",
            "hs2",
            "product_description",
            "total_trade",
            "n_exporters",
            "effective_n_exporters",
            "dominant_supplier",
            "dominant_supplier_share",
            "dominant_supplier_slack",
            "dominant_supplier_risk",
            "HHI",
            "Slack_min",
            "n_critical_suppliers",
            "n_severe_suppliers",
            "critical_supplier_share",
            "product_fragility_score",
            "max_criticality_depth"
        ]
    ].head(10)
)

# ============================================================
# 13B. SYSTEMICALLY RELEVANT FRAGILE PRODUCTS
# ============================================================

# Pure fragility may over-rank small or niche products.
# This alternative ranking combines structural fragility with trade scale.

systemic_fragile_products = top_fragile_products.copy()

systemic_fragile_products["rank_fragility"] = (
    systemic_fragile_products["product_fragility_score"]
    .rank(method="min", ascending=False)
)

systemic_fragile_products["rank_trade"] = (
    systemic_fragile_products["total_trade"]
    .rank(method="min", ascending=False)
)

systemic_fragile_products["systemic_fragility_rank"] = (
    systemic_fragile_products["rank_fragility"]
    + systemic_fragile_products["rank_trade"]
)

systemic_fragile_products = systemic_fragile_products.sort_values(
    [
        "systemic_fragility_rank",
        "product_fragility_score",
        "total_trade"
    ],
    ascending=[True, False, False]
)

print("\nSystemically relevant fragile products:")
display(
    systemic_fragile_products[
        [
            "year",
            "hs4",
            "hs2",
            "product_description",
            "total_trade",
            "n_exporters",
            "effective_n_exporters",
            "dominant_supplier",
            "dominant_supplier_share",
            "dominant_supplier_slack",
            "dominant_supplier_risk",
            "HHI",
            "Slack_min",
            "n_critical_suppliers",
            "critical_supplier_share",
            "product_fragility_score",
            "rank_fragility",
            "rank_trade",
            "systemic_fragility_rank"
        ]
    ].head(10)
)

### 13. Top fragile products

This cell identifies HS4 products with at least one pivotal supplier and ranks them from two complementary perspectives. The first table ranks products by the threshold-sensitive `product_fragility_score`, which combines the supplier’s market share with the depth of criticality, \(\max(0,1-Slack_{cp})\). This ranking captures pure structural fragility: products at the top are those where one supplier accounts for a very large share of observed supply and where the removal of that supplier would create a large shortfall relative to the quota.

The second table ranks products by systemic fragility, combining the structural fragility ranking with total trade value. This is necessary because the most fragile products are not always the most economically relevant. Some top-ranked products by pure fragility, such as cocoa waste or jute yarn, show near-monopolistic supplier structures but have relatively limited trade value. By contrast, the systemic ranking gives more weight to products that combine meaningful fragility with larger trade flows, such as toys and vehicles for children, electric lighting equipment, soybeans, iron ores, watches, lignite, and selected manufactured goods.

HHI is included as a benchmark concentration measure, but it is not the main ranking criterion. The main fragility score is threshold-sensitive: it activates only when a supplier crosses the pivotality condition \(Slack_{cp}<1\). Therefore, this table should be interpreted as a bridge between standard concentration analysis and quota-based structural dependence.

The results show that 459 out of 1,222 HS4 products have at least one critical supplier. The pure fragility ranking is dominated by products with very high dominant-supplier shares, often above 80%, and very low effective numbers of exporters. These cases validate the metric because they identify product markets that are structurally close to single-supplier dependence.

The systemic ranking changes the interpretation by incorporating trade scale. Products such as HS9503, HS9405, HS9504, HS1201, and HS2601 become more relevant because they combine non-negligible fragility with larger trade volumes. This ranking is more suitable for the paper’s policy discussion, since it connects structural dependence to economically meaningful supply chains.

In [ ]:
# ============================================================
# 14. VISUAL DIAGNOSTICS
# ============================================================

# ------------------------------------------------------------
# 14A. Top pivotal supplier countries by portfolio weighted slack
# ------------------------------------------------------------

top_countries_plot = top_pivotal_countries_by_slack.head(20).copy()

plt.figure(figsize=(10, 6))
plt.barh(
    top_countries_plot["country_name"][::-1],
    top_countries_plot["Slack_weighted"][::-1]
)
plt.title(f"Top Pivotal Supplier Countries by Portfolio Weighted Slack, {YEAR}")
plt.xlabel("Weighted slack")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 14B. Top fragile products by threshold-sensitive fragility score
# ------------------------------------------------------------

top_products_plot = top_fragile_products.head(20).copy()

product_labels = (
    top_products_plot["hs4"].astype(str)
    + " - "
    + top_products_plot["product_description"].astype(str).str.slice(0, 45)
)

plt.figure(figsize=(10, 7))
plt.barh(
    product_labels[::-1],
    top_products_plot["product_fragility_score"][::-1]
)
plt.title(f"Most Fragile Products by Threshold-Sensitive Fragility Score, {YEAR}")
plt.xlabel("Product fragility score")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 14C. Systemically relevant fragile products
# ------------------------------------------------------------

systemic_products_plot = systemic_fragile_products.head(20).copy()

systemic_product_labels = (
    systemic_products_plot["hs4"].astype(str)
    + " - "
    + systemic_products_plot["product_description"].astype(str).str.slice(0, 45)
)

plt.figure(figsize=(10, 7))
plt.barh(
    systemic_product_labels[::-1],
    systemic_products_plot["systemic_fragility_rank"][::-1]
)
plt.title(f"Systemically Relevant Fragile Products, {YEAR}")
plt.xlabel("Systemic fragility rank")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 14D. HHI versus threshold-sensitive product fragility
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))
plt.scatter(
    product_slack["HHI"],
    product_slack["product_fragility_score"],
    alpha=0.5
)
plt.title(f"HHI vs Threshold-Sensitive Product Fragility, {YEAR}")
plt.xlabel("HHI")
plt.ylabel("Product fragility score")
plt.tight_layout()
plt.show()


# Optional diagnostic: correlation between HHI and product fragility score
corr_hhi_fragility = product_slack[["HHI", "product_fragility_score"]].corr().iloc[0, 1]
print("Correlation between HHI and product fragility score:", round(corr_hhi_fragility, 4))

### 14. Visual diagnostics

This cell provides graphical diagnostics for the main country-level and product-level results. The first figure ranks exporter countries by portfolio weighted slack. Lower values indicate countries whose economically important export products are closer to, or below, the pivotality threshold. This figure should be interpreted as a measure of portfolio-level pivotality intensity, not as a general ranking of global economic size.

The second figure shows the most fragile HS4 products according to the threshold-sensitive product fragility score. This ranking captures pure structural fragility: products at the top are those where one supplier accounts for a very large share of observed supply and where the removal of that supplier would create a large shortfall relative to the quota.

The third figure focuses on systemically relevant fragile products. This view combines structural fragility with trade scale, making it more suitable for policy interpretation. It helps distinguish highly fragile but economically small products from products that are both fragile and relevant for global supply chains.

The final scatter plot compares HHI with the threshold-sensitive product fragility score. The correlation is high, indicating that the proposed fragility measure is strongly related to supplier concentration. However, the two measures are not conceptually identical. HHI measures concentration across all suppliers, while the slack-based fragility score activates only when a supplier crosses the pivotality threshold \(Slack_{cp}<1\). Therefore, the proposed metric should be interpreted as a threshold-sensitive refinement of concentration analysis rather than as a completely independent alternative to HHI.

The visual diagnostics confirm three main points. First, the country-level weighted slack ranking highlights portfolio-level pivotality: countries such as Guinea, China, Australia, Zambia, Brazil, and Indonesia have export baskets in which important products are structurally close to the pivotality threshold. Second, the pure product-fragility ranking is dominated by products with very high supplier concentration, such as cocoa waste, jute yarn, artificial flowers, lignite, vacuum flasks, and similar products. Third, the systemic product ranking changes the emphasis by incorporating trade scale, bringing forward products such as toys, lighting equipment, soybeans, iron ores, watches, lignite, and manufactured goods.

The HHI-fragility scatter plot shows a strong positive correlation of 0.9481. This is expected because both indicators are related to supplier concentration. However, the slack-based score introduces a pivotality threshold: products without a supplier exceeding the critical share have zero fragility, while products with pivotal suppliers are ranked according to the depth of the quota violation. For the paper, this means that HHI remains an important benchmark, but the slack framework adds an explicit feasibility interpretation.

In [ ]:
# ============================================================
# 14A. CRITICALITY CONTRIBUTION
# ============================================================

# Edge-level criticality depth:
#   criticality_depth = max(0, 1 - Slack_cp)
#
# This is positive only when Slack_cp < 1.
# Therefore, it captures how far below the quota the product market
# would fall if the supplier were removed.

edges_cp["criticality_depth"] = np.maximum(
    0,
    1 - edges_cp["Slack_cp"]
)

# Country-level contribution to pivotality:
#   beta_cp captures how important product p is in country c's export portfolio.
#   criticality_depth captures how pivotal country c is for product p.
#
# This answers:
#   "Which products make country c structurally pivotal?"

edges_cp["country_criticality_contribution"] = (
    edges_cp["beta_cp"] * edges_cp["criticality_depth"]
)

# Keep only links with positive criticality contribution
criticality_contribution_edges = (
    edges_cp
    .query("country_criticality_contribution > 0")
    .sort_values(
        ["country_criticality_contribution", "weight", "supply_share_cp"],
        ascending=[False, False, False]
    )
    .copy()
)

print("Number of links with positive country criticality contribution:",
      criticality_contribution_edges.shape[0])

display(
    criticality_contribution_edges[
        [
            "year",
            "country_name",
            "hs4",
            "hs2",
            "product_description",
            "weight",
            "supply_share_cp",
            "Slack_cp",
            "beta_cp",
            "criticality_depth",
            "country_criticality_contribution",
            "risk_label"
        ]
    ].head(30)
)

# Optional: inspect the products that make a specific country pivotal
COUNTRY_TO_INSPECT = "Italy"

country_profile = (
    criticality_contribution_edges
    .query("country_name == @COUNTRY_TO_INSPECT")
    .sort_values(
        ["country_criticality_contribution", "weight"],
        ascending=[False, False]
    )
)

print(f"\nTop products contributing to {COUNTRY_TO_INSPECT}'s criticality:")
display(
    country_profile[
        [
            "year",
            "country_name",
            "hs4",
            "hs2",
            "product_description",
            "weight",
            "supply_share_cp",
            "Slack_cp",
            "beta_cp",
            "criticality_depth",
            "country_criticality_contribution",
            "risk_label"
        ]
    ].head(20)
)

### 14A. Criticality contribution and Italy’s export-side pivotality

This cell decomposes country-level pivotality into product-specific contributions. The variable `criticality_depth` is defined as \(\max(0,1-Slack_{cp})\), so it is positive only when a country-product link is critical. The variable `country_criticality_contribution` multiplies this depth by \(\beta_{cp}\), the share of product \(p\) in country \(c\)'s own export portfolio. A high value therefore means that the country is pivotal in a product market and that the product is also relevant within the country’s export structure.

This measure answers the question: which products make a country structurally pivotal as a supplier? It is different from the product-level fragility ranking, which asks which HS4 products are globally fragile. It is also different from an import-vulnerability analysis, because here Italy is treated as an exporter. The Italy-specific table therefore identifies the products for which Italy acts as a critical supplier in the global HS4 network.

Including Italy is useful for the paper because the event is held in Italy and the policy discussion concerns strategic autonomy and trade exposure. The results can be interpreted as the export-side complement to import vulnerability: Italy may be exposed to foreign pivotal suppliers in some sectors, but it may also itself be pivotal in selected product markets, such as tomato preparations, wool fabrics, leather products, vinegar, vermouth, and machinery for leather processing.

In [ ]:
# ============================================================
# 14B. MOST CRITICAL PRODUCTS OF A GIVEN COUNTRY
# ============================================================

def find_country_name(edges, query):
    """
    Finds the closest country name available in edges_cp.
    """
    countries = sorted(edges["country_name"].dropna().unique())

    exact = [c for c in countries if c.lower() == query.lower()]
    if exact:
        return exact[0]

    partial = [c for c in countries if query.lower() in c.lower()]

    if len(partial) == 0:
        raise ValueError(f"No country found for query: {query}")

    print("Possible matches:")
    for c in partial[:20]:
        print("-", c)

    return partial[0]


def critical_products_for_country(edges, country_query, top_n=30, only_critical=True):
    """
    Returns the most critical products for a selected exporter country.

    Interpretation:
      These are products for which the selected country is a relevant supplier.
      If only_critical=True, the table keeps only products where Slack_cp < 1.
    """

    country = find_country_name(edges, country_query)
    print(f"\nSelected country: {country}")

    df = edges[edges["country_name"] == country].copy()

    # Make sure required variables exist
    if "criticality_depth" not in df.columns:
        df["criticality_depth"] = np.maximum(0, 1 - df["Slack_cp"])

    if "country_criticality_contribution" not in df.columns:
        df["country_criticality_contribution"] = (
            df["beta_cp"] * df["criticality_depth"]
        )

    if only_critical:
        df = df[df["Slack_cp"] < 1].copy()

    if df.empty:
        print("No critical products found for this country under the current theta and filters.")
        return df

    df = df.sort_values(
        [
            "country_criticality_contribution",
            "criticality_depth",
            "supply_share_cp",
            "weight"
        ],
        ascending=[False, False, False, False]
    )

    cols = [
        "year",
        "country_name",
        "hs4",
        "hs2",
        "product_description",
        "weight",
        "T_p",
        "supply_share_cp",
        "Slack_cp",
        "beta_cp",
        "criticality_depth",
        "country_criticality_contribution",
        "risk_label"
    ]

    return df[cols].head(top_n)


# Example: Italy as global supplier
IT_critical_products = critical_products_for_country(
    edges_cp,
    country_query="Italy",
    top_n=10,
    only_critical=True
)

IT_critical_products

### 14B. Country-specific critical product profile

This cell provides a country-specific drill-down of the criticality contribution measure computed in the previous step. While the previous table ranks all critical exporter-product links globally, this function allows the analysis to focus on one selected exporter country. For a given country, it returns the products for which the country is a critical supplier, ranked by `country_criticality_contribution`.

This is useful for interpreting the country-level results. In the case of China, the table identifies the HS4 products for which China acts as a pivotal supplier in the global product network. This should be interpreted as an export-side pivotality profile, not as an import-vulnerability measure. The import-side analysis of China requires a separate importer-based network.

In [ ]:
# ============================================================
# 14C. BIPARTITE VISUALIZATION
# ============================================================

import networkx as nx
import textwrap
import matplotlib.pyplot as plt


def short_product_label(hs4, description, max_chars=45):
    desc = str(description)
    desc = desc.replace("\n", " ")
    if len(desc) > max_chars:
        desc = desc[:max_chars] + "..."
    return f"{hs4} | {desc}"


def plot_country_product_bipartite(
    edges,
    focal_country_query="Italy",
    top_products=8,
    top_exporters_per_product=6,
    only_critical_products=True,
    figsize=(14, 9)
):
    """
    Creates a focused country-product bipartite graph.

    Left side:
      Supplier countries.

    Right side:
      Products.

    Red edges:
      Critical country-product edges, Slack_cp < 1.

    Gray edges:
      Non-critical but relevant supplier-product edges.
    """

    focal_country = find_country_name(edges, focal_country_query)
    print(f"\nFocal country: {focal_country}")

    focal_edges = edges[edges["country_name"] == focal_country].copy()

    if only_critical_products:
        focal_edges = focal_edges[focal_edges["Slack_cp"] < 1].copy()

    if focal_edges.empty:
        raise ValueError(
            f"No products found for {focal_country} under the selected filter."
        )

    # Select products where the focal country is most pivotal
    selected_products = (
        focal_edges
        .sort_values(
            [
                "country_criticality_contribution",
                "criticality_depth",
                "supply_share_cp",
                "weight"
            ],
            ascending=[False, False, False, False]
        )
        .head(top_products)["hs4"]
        .unique()
        .tolist()
    )

    # For those products, keep the top exporters
    sub_parts = []

    for product in selected_products:
        tmp = edges[edges["hs4"] == product].copy()

        # Always include the focal country
        focal_tmp = tmp[tmp["country_name"] == focal_country]

        # Main suppliers by supply share
        top_tmp = (
            tmp
            .sort_values(["supply_share_cp", "weight"], ascending=[False, False])
            .head(top_exporters_per_product)
        )

        tmp_keep = pd.concat([focal_tmp, top_tmp], ignore_index=True)
        tmp_keep = tmp_keep.drop_duplicates(subset=["country_name", "hs4"])
        sub_parts.append(tmp_keep)

    sub = pd.concat(sub_parts, ignore_index=True)

    # Product labels
    product_label_map = (
        sub[["hs4", "product_description"]]
        .drop_duplicates("hs4")
        .assign(
            product_label=lambda d: d.apply(
                lambda row: short_product_label(row["hs4"], row["product_description"]),
                axis=1
            )
        )
        .set_index("hs4")["product_label"]
        .to_dict()
    )

    sub["product_label"] = sub["hs4"].map(product_label_map)

    # Build graph
    G = nx.Graph()

    countries = sorted(sub["country_name"].unique())
    products = [product_label_map[h] for h in selected_products if h in product_label_map]

    for c in countries:
        G.add_node(c, bipartite="country")

    for p in products:
        G.add_node(p, bipartite="product")

    for _, row in sub.iterrows():
        G.add_edge(
            row["country_name"],
            row["product_label"],
            weight=row["weight"],
            supply_share=row["supply_share_cp"],
            slack=row["Slack_cp"],
            critical=row["Slack_cp"] < 1,
            risk_label=row["risk_label"]
        )

    # Manual bipartite layout
    countries_sorted = sorted(
        countries,
        key=lambda c: 0 if c == focal_country else 1
    )

    products_sorted = products

    pos = {}

    if len(countries_sorted) == 1:
        pos[countries_sorted[0]] = (0, 0)
    else:
        for idx, c in enumerate(countries_sorted):
            y = 1 - 2 * idx / (len(countries_sorted) - 1)
            pos[c] = (0, y)

    if len(products_sorted) == 1:
        pos[products_sorted[0]] = (1, 0)
    else:
        for idx, p in enumerate(products_sorted):
            y = 1 - 2 * idx / (len(products_sorted) - 1)
            pos[p] = (1, y)

    # Edge styles
    edge_colors = [
        "crimson" if G[u][v]["critical"] else "lightgray"
        for u, v in G.edges()
    ]

    edge_widths = [
        0.5 + 6 * G[u][v]["supply_share"]
        for u, v in G.edges()
    ]

    country_node_colors = [
        "gold" if node == focal_country else "skyblue"
        for node in countries_sorted
    ]

    product_node_colors = ["lightgreen" for _ in products_sorted]

    plt.figure(figsize=figsize)

    nx.draw_networkx_edges(
        G,
        pos,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.75
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=countries_sorted,
        node_color=country_node_colors,
        node_size=900,
        edgecolors="black"
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=products_sorted,
        node_color=product_node_colors,
        node_size=1100,
        edgecolors="black",
        node_shape="s"
    )

    labels = {}
    for node in G.nodes():
        if node in products_sorted:
            labels[node] = "\n".join(textwrap.wrap(node, 35))
        else:
            labels[node] = node

    nx.draw_networkx_labels(
        G,
        pos,
        labels=labels,
        font_size=8
    )

    plt.title(
        f"Country–Product Bipartite Slack Network: {focal_country}\n"
        f"Red edges = critical supplier-product links (Slack < 1)"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return sub.sort_values(
        ["hs4", "supply_share_cp"],
        ascending=[True, False]
    )


# Example visualization for Italy
italy_bipartite_subgraph = plot_country_product_bipartite(
    edges_cp,
    focal_country_query="Italy",
    top_products=8,
    top_exporters_per_product=6,
    only_critical_products=True,
    figsize=(15, 9)
)

italy_bipartite_subgraph[
    [
        "country_name",
        "hs4",
        "product_description",
        "weight",
        "supply_share_cp",
        "Slack_cp",
        "risk_label"
    ]
].head(50)

### 14C. Bipartite visualization of Italy’s export-side pivotality

This cell creates a focused bipartite network for a selected exporter country. In this case, Italy is used as the focal country. The function first identifies the HS4 products for which Italy is most structurally pivotal, based on `country_criticality_contribution`, and then adds the main competing suppliers for those same products.

The left side of the graph represents supplier countries, while the right side represents HS4 products. Italy is highlighted as the focal supplier. Red edges indicate critical supplier-product links, defined by \(Slack_{cp}<1\), while gray edges indicate relevant but non-critical supplier-product links. Edge thickness is proportional to the supplier’s share in the product market.

This visualization should be interpreted as an export-side pivotality profile. It shows the products for which Italy acts as a critical supplier in the global HS4 network, not the products for which Italy is dependent on foreign suppliers. In the Italy example, the critical products include tomato preparations, wool fabrics, leather products, vinegar, vermouth, and machinery for leather processing. The graph also shows the main alternative suppliers in those product markets, making it possible to see whether Italy’s pivotality occurs in relatively concentrated or more diversified supplier structures.

This figure is mainly exploratory and useful for communicating the bipartite logic of the method. For the paper, it can support the interpretation that Italy is not only exposed to global supply dependencies, but also occupies structurally important supplier positions in selected product markets. A separate importer-side analysis is required to study Italy’s strategic import vulnerability.

In [ ]:
# ============================================================
# 14D. IMPORT VULNERABILITY FOR A GIVEN COUNTRY
# ============================================================

def get_country_code(country_lookup, query):
    """
    Finds the country code for a selected importer country.
    It first searches for an exact match, then for partial matches.
    """

    # Exact match first
    exact = country_lookup[
        country_lookup["country_name"].str.lower() == query.lower()
    ].copy()

    if not exact.empty:
        print("Exact match found:")
        display(exact.head(20))
        return int(exact.iloc[0]["country_code"])

    # Partial match if exact match is not found
    matches = country_lookup[
        country_lookup["country_name"].str.contains(query, case=False, na=False)
    ].copy()

    if matches.empty:
        raise ValueError(f"No country code found for query: {query}")

    print("Partial matches found:")
    display(matches.head(20))

    return int(matches.iloc[0]["country_code"])


def importer_product_vulnerability(
    baci_file,
    country_lookup,
    importer_query="Italy",
    theta=0.70,
    min_weight=100,
    chunksize=1_000_000
):
    """
    For a selected importing country j, build a supplier-product network:

        supplier country c -> HS4 product p -> importer country j

    Edge weight:
        imports of product p by importer j from supplier c.

    Import slack:
        Slack_import_cp = (T_import_p - w_cp) / (theta * T_import_p)

    Interpretation:
        Slack_import_cp < 1 means that supplier c is pivotal for importer j
        in product p. If supplier c disappears, the importer would retain
        less than the required quota of product p imports.

    With theta = 0.70:
        Slack_import_cp < 1 is equivalent to supplier_share > 0.30.
    """

    importer_code = get_country_code(country_lookup, importer_query)

    chunks = []

    for chunk in pd.read_csv(
        baci_file,
        usecols=["t", "i", "j", "k", "v"],
        chunksize=chunksize
    ):

        # Keep only imports of the selected country
        chunk = chunk[chunk["j"] == importer_code].copy()

        if chunk.empty:
            continue

        # Convert trade values before filtering
        chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce")

        chunk = chunk.dropna(subset=["i", "k", "v"])
        chunk = chunk[chunk["v"] > 0].copy()

        if chunk.empty:
            continue

        # Standardize supplier and product codes
        chunk["i"] = chunk["i"].astype(int)

        chunk["k"] = (
            chunk["k"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(6)
        )

        chunk["hs4"] = chunk["k"].str[:4]
        chunk["hs2"] = chunk["k"].str[:2]

        # Aggregate inside chunk: supplier country x HS4 product
        agg = (
            chunk
            .groupby(["i", "hs4", "hs2"], as_index=False)
            .agg(
                weight=("v", "sum"),
                n_records=("v", "size")
            )
        )

        chunks.append(agg)

    if len(chunks) == 0:
        raise ValueError("No import records found for this country.")

    imports = pd.concat(chunks, ignore_index=True)

    # Aggregate again across chunks
    imports = (
        imports
        .groupby(["i", "hs4", "hs2"], as_index=False)
        .agg(
            weight=("weight", "sum"),
            n_records=("n_records", "sum")
        )
    )

    # Keep economically meaningful import links
    imports = imports[imports["weight"] >= min_weight].copy()

    # Add supplier country names
    imports = imports.merge(
        country_lookup,
        left_on="i",
        right_on="country_code",
        how="left"
    )

    imports = imports.rename(columns={"country_name": "supplier_country"})

    imports["supplier_country"] = imports["supplier_country"].fillna(
        "C_" + imports["i"].astype(str)
    )

    imports = imports.drop(columns=["country_code"])

    # Add product descriptions from the global edge table
    if "product_description" in edges_cp.columns:
        product_desc_lookup = (
            edges_cp[["hs4", "product_description"]]
            .drop_duplicates("hs4")
        )
        imports = imports.merge(product_desc_lookup, on="hs4", how="left")
    else:
        imports["product_description"] = imports["hs4"]

    imports["product_description"] = imports["product_description"].fillna(imports["hs4"])

    imports["year"] = YEAR
    imports["importer_country"] = importer_query

    # Total imports of product p by selected importer
    imports["T_import_p"] = imports.groupby("hs4")["weight"].transform("sum")

    # Import quota
    imports["Q_import_p"] = theta * imports["T_import_p"]

    # Supplier share in the selected importer's product-specific import basket
    imports["supplier_share"] = imports["weight"] / imports["T_import_p"]

    # Import-side slack
    imports["Slack_import_cp"] = (
        imports["T_import_p"] - imports["weight"]
    ) / imports["Q_import_p"]

    # Gap relative to pivotality threshold
    imports["import_slack_gap"] = imports["Slack_import_cp"] - 1

    # Criticality indicator
    imports["is_critical_import_supplier"] = imports["Slack_import_cp"] < 1

    # Depth of import criticality
    imports["import_criticality_depth"] = np.maximum(
        0,
        1 - imports["Slack_import_cp"]
    )

    # Risk labels adapted to the bounded slack scale
    imports["risk_label"] = np.select(
        [
            imports["Slack_import_cp"] < 0.5,
            imports["Slack_import_cp"] < 1.0,
            imports["Slack_import_cp"] < 1.1,
            imports["Slack_import_cp"] < 1.2
        ],
        [
            "SEVERE",
            "CRITICAL",
            "NEAR-CRITICAL",
            "WATCH"
        ],
        default="REDUNDANT"
    )

    # Route vulnerability score:
    # combines import value, supplier share, and depth of criticality.
    imports["route_vulnerability_score"] = (
        imports["supplier_share"] *
        imports["import_criticality_depth"] *
        np.log1p(imports["weight"])
    )

    imports = imports.sort_values(
        [
            "Slack_import_cp",
            "supplier_share",
            "weight"
        ],
        ascending=[True, False, False]
    )

    return imports


# Example: Italy as importer
italy_import_vulnerability = importer_product_vulnerability(
    baci_file=baci_file,
    country_lookup=country_lookup,
    importer_query="Italy",
    theta=THETA,
    min_weight=100
)

print("========== ITALY IMPORT VULNERABILITY ==========")
print("Importer:", "Italy")
print("Number of supplier-product import links:", italy_import_vulnerability.shape[0])
print("Number of supplier countries:", italy_import_vulnerability["supplier_country"].nunique())
print("Number of HS4 products imported:", italy_import_vulnerability["hs4"].nunique())

print("\nCritical import links:")
print(italy_import_vulnerability["is_critical_import_supplier"].sum())

print("\nShare of critical import links:")
print(
    round(
        100 * italy_import_vulnerability["is_critical_import_supplier"].mean(),
        2
    ),
    "%"
)

print("\nRisk label distribution:")
print(italy_import_vulnerability["risk_label"].value_counts())

display(
    italy_import_vulnerability[
        [
            "year",
            "importer_country",
            "supplier_country",
            "hs4",
            "hs2",
            "product_description",
            "weight",
            "T_import_p",
            "supplier_share",
            "Slack_import_cp",
            "import_criticality_depth",
            "risk_label",
            "route_vulnerability_score"
        ]
    ].head(40)
)

### 14D. Import vulnerability for Italy

This cell extends the slack framework to the importer side. Instead of asking whether a country is pivotal as a global supplier, the analysis asks whether Italy depends critically on specific foreign suppliers. For Italy as importer, the code builds a supplier-product network where each edge represents imports of HS4 product \(p\) from supplier country \(c\).

For each imported product, total imports are defined as \(T^{imp}_p=\sum_c w_{cp}\), and the quota is \(Q^{imp}_p=\theta T^{imp}_p\). Import slack is then defined as:

\[
Slack^{imp}_{cp}=\frac{T^{imp}_p-w_{cp}}{\theta T^{imp}_p}.
\]

A supplier is critical for Italy when \(Slack^{imp}_{cp}<1\). With \(\theta=0.70\), this is equivalent to the supplier accounting for more than 30% of Italy’s observed imports of that HS4 product. In this sense, the metric identifies import routes where Italy has limited short-run supplier redundancy.

The diagnostic results show that Italy’s thresholded import network contains 29,043 supplier-product links, involving 182 supplier countries and 1,209 imported HS4 products. Among these links, 815 are critical, corresponding to 2.81% of the network. The risk classification identifies 97 severe import links, 718 critical links, 434 near-critical links, 750 watch links, and 27,044 redundant links.

The first rows include several cases where Italy imports a product entirely from one supplier within the thresholded network, producing \(Slack^{imp}_{cp}=0\). These are structurally severe cases, but they should be interpreted carefully because some involve relatively small trade values. For policy interpretation, the route vulnerability score is useful because it combines supplier share, depth of criticality, and trade value. This helps distinguish small single-supplier routes from economically more relevant dependencies.

The Italy import-side results are central for the paper’s strategic-autonomy argument. They identify foreign supplier-product routes where Italy is structurally exposed, such as selected mineral inputs, dredgers, nuclear reactors, plants and flowers, iron and steel inputs, cobalt ores, nickel mattes, and other specialized products. These routes measure structural dependence, not geopolitical risk directly. Geopolitical vulnerability emerges when this dependence overlaps with strategic sectors, political risk, logistical disruption, or low substitutability.

In [ ]:
# ============================================================
# 14F. HS2-LEVEL SUMMARY FOR POSTER TABLES
# ============================================================

# This table aggregates HS4 product-level fragility to HS2 chapters.
# It is useful for sectoral interpretation and poster/paper summary tables.
#
# Important:
#   total_product_fragility can be higher in HS2 chapters with many HS4 products.
#   Therefore, we also report average fragility and the share of products
#   with at least one critical supplier.

hs2_fragility_summary = (
    product_slack
    .groupby(["year", "hs2"], as_index=False)
    .agg(
        n_products=("hs4", "nunique"),
        total_trade=("total_trade", "sum"),

        # Fragility indicators
        avg_product_fragility=("product_fragility_score", "mean"),
        median_product_fragility=("product_fragility_score", "median"),
        max_product_fragility=("product_fragility_score", "max"),
        total_product_fragility=("product_fragility_score", "sum"),

        # Critical supplier structure
        avg_critical_supplier_share=("critical_supplier_share", "mean"),
        n_products_with_critical_suppliers=("n_critical_suppliers", lambda x: (x > 0).sum()),
        total_critical_suppliers=("n_critical_suppliers", "sum"),

        # Concentration benchmarks
        avg_HHI=("HHI", "mean"),
        max_supplier_share=("max_supply_share", "max")
    )
)

hs2_fragility_summary["share_products_with_critical_suppliers"] = (
    hs2_fragility_summary["n_products_with_critical_suppliers"]
    / hs2_fragility_summary["n_products"]
)

# Ranking variables
hs2_fragility_summary["rank_total_fragility"] = (
    hs2_fragility_summary["total_product_fragility"]
    .rank(method="min", ascending=False)
)

hs2_fragility_summary["rank_avg_fragility"] = (
    hs2_fragility_summary["avg_product_fragility"]
    .rank(method="min", ascending=False)
)

hs2_fragility_summary["rank_trade"] = (
    hs2_fragility_summary["total_trade"]
    .rank(method="min", ascending=False)
)

hs2_fragility_summary["rank_critical_share"] = (
    hs2_fragility_summary["share_products_with_critical_suppliers"]
    .rank(method="min", ascending=False)
)

# Combined descriptive sector ranking
hs2_fragility_summary["combined_hs2_fragility_rank"] = (
    hs2_fragility_summary["rank_total_fragility"]
    + hs2_fragility_summary["rank_trade"]
    + hs2_fragility_summary["rank_critical_share"]
)

hs2_fragility_summary = hs2_fragility_summary.sort_values(
    [
        "combined_hs2_fragility_rank",
        "total_product_fragility",
        "share_products_with_critical_suppliers",
        "total_trade"
    ],
    ascending=[True, False, False, False]
)

display(
    hs2_fragility_summary[
        [
            "year",
            "hs2",
            "n_products",
            "total_trade",
            "n_products_with_critical_suppliers",
            "share_products_with_critical_suppliers",
            "total_critical_suppliers",
            "total_product_fragility",
            "avg_product_fragility",
            "median_product_fragility",
            "max_product_fragility",
            "avg_critical_supplier_share",
            "avg_HHI",
            "max_supplier_share",
            "rank_total_fragility",
            "rank_avg_fragility",
            "rank_trade",
            "rank_critical_share",
            "combined_hs2_fragility_rank"
        ]
    ].head(30)
)

### 14F. HS2-level fragility summary

This cell aggregates HS4 product-level fragility measures into broader HS2 chapters. The objective is to move from individual product markets to sector-level patterns that are easier to interpret in a paper or poster. For each HS2 chapter, the table reports the number of HS4 products, total trade value, the number and share of products with at least one critical supplier, total product fragility, average product fragility, HHI, and maximum supplier share.

The sectoral ranking combines three dimensions: cumulative fragility, trade scale, and the share of HS4 products with critical suppliers. This is useful because HS2 chapters differ substantially in size. A sector may have high total fragility because it contains many HS4 products, while another may have high average fragility because a smaller number of products are highly concentrated. Therefore, both total and average measures are reported.

The results show that HS2 chapter 85, electrical machinery and equipment, ranks first in the combined sectoral ranking. It combines very large trade value with a high number of fragile HS4 products: 23 out of 46 products have at least one critical supplier. Other relevant chapters include HS95, toys and sports-related products; HS29, organic chemicals; HS26, ores and concentrates; HS96, miscellaneous manufactured articles; HS28, inorganic chemicals; HS82, tools and cutlery; and HS71, precious metals and related products.

This table is useful for the paper because it connects the product-level slack results to broader sectoral patterns. It shows that fragility is not confined to niche products: some high-trade sectors, especially electrical machinery, chemicals, minerals, metals, and selected manufactured goods, also contain multiple products with pivotal supplier structures. The combined HS2 ranking should be interpreted as descriptive rather than definitive, but it provides a compact sectoral view of where supply-chain fragility is concentrated.

In [ ]:
# ============================================================
# 14G. POSTER VISUAL 1
# QUESTION:
#   "From which countries does Italy depend on critical imported products?"
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install only if needed. In Colab, this may take a few seconds.
!pip -q install plotly country_converter kaleido

import plotly.express as px
import country_converter as coco
from IPython.display import display, Image, HTML

QUESTION_IMPORT_ROUTES = (
    "From which countries does Italy depend on critical imported products?"
)

print(QUESTION_IMPORT_ROUTES)

# Output folders
OUT_DIR = os.path.join(DATA_DIR, "results_country_product_weighted_slack")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TABLE_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)


# ------------------------------------------------------------
# HS2 macro-section labels
# ------------------------------------------------------------

def assign_hs2_section(hs2):
    hs2 = str(hs2).zfill(2)
    x = int(hs2)

    if 1 <= x <= 5:
        return "Animals"
    elif 6 <= x <= 14:
        return "Vegetable products"
    elif x == 15:
        return "Fats & oils"
    elif 16 <= x <= 24:
        return "Food & beverages"
    elif 25 <= x <= 27:
        return "Minerals & fuels"
    elif 28 <= x <= 38:
        return "Chemicals"
    elif 39 <= x <= 40:
        return "Plastics & rubber"
    elif 41 <= x <= 43:
        return "Leather"
    elif 44 <= x <= 49:
        return "Wood & paper"
    elif 50 <= x <= 63:
        return "Textiles"
    elif 64 <= x <= 67:
        return "Footwear"
    elif 68 <= x <= 71:
        return "Stone, glass, precious metals"
    elif 72 <= x <= 83:
        return "Base metals"
    elif 84 <= x <= 85:
        return "Machinery & electronics"
    elif 86 <= x <= 89:
        return "Transport equipment"
    elif 90 <= x <= 99:
        return "Other manufactured goods"
    else:
        return "Other"


# ------------------------------------------------------------
# Build critical route table
# ------------------------------------------------------------

italy_routes = italy_import_vulnerability.copy()

italy_routes["hs2"] = italy_routes["hs2"].astype(str).str.zfill(2)
italy_routes["hs4"] = italy_routes["hs4"].astype(str).str.zfill(4)
italy_routes["hs2_section"] = italy_routes["hs2"].apply(assign_hs2_section)

# Keep only critical routes
italy_critical_routes = italy_routes[
    italy_routes["Slack_import_cp"] < 1
].copy()

# Depth of criticality: how far below the critical threshold the route is
italy_critical_routes["criticality_depth"] = (
    1 - italy_critical_routes["Slack_import_cp"]
)

# BACI values are usually in thousand USD.
# weight / 1000 gives approximately million USD.
italy_critical_routes["trade_value_musd"] = (
    italy_critical_routes["weight"] / 1000
)

# Combined vulnerability score
italy_critical_routes["route_vulnerability_score"] = (
    italy_critical_routes["criticality_depth"]
    * italy_critical_routes["supplier_share"]
    * np.log1p(italy_critical_routes["weight"])
)

italy_critical_routes = italy_critical_routes.sort_values(
    ["route_vulnerability_score", "supplier_share", "weight"],
    ascending=[False, False, False]
)

italy_critical_route_table = italy_critical_routes[
    [
        "year",
        "importer_country",
        "supplier_country",
        "hs4",
        "hs2",
        "hs2_section",
        "product_description",
        "weight",
        "trade_value_musd",
        "T_import_p",
        "supplier_share",
        "Slack_import_cp",
        "criticality_depth",
        "route_vulnerability_score",
        "risk_label"
    ]
].copy()

print("Number of critical import routes:", italy_critical_route_table.shape[0])
display(italy_critical_route_table.head(30))

# Save route table
route_table_path = os.path.join(TABLE_DIR, f"italy_critical_import_routes_{YEAR}.csv")
italy_critical_route_table.to_csv(route_table_path, index=False)
print("Saved route table to:", route_table_path)


# ------------------------------------------------------------
# Aggregate by supplier country for map and bar chart
# ------------------------------------------------------------

supplier_summary = (
    italy_critical_routes
    .groupby("supplier_country", as_index=False)
    .agg(
        n_critical_routes=("hs4", "nunique"),
        critical_trade_value_musd=("trade_value_musd", "sum"),
        total_route_vulnerability_score=("route_vulnerability_score", "sum"),
        avg_supplier_share=("supplier_share", "mean"),
        max_supplier_share=("supplier_share", "max"),
        min_import_slack=("Slack_import_cp", "min")
    )
)

# Add ISO3 code if available from the import vulnerability table
if "country_iso3" in italy_critical_routes.columns:
    iso_lookup = (
        italy_critical_routes[["supplier_country", "country_iso3"]]
        .drop_duplicates("supplier_country")
    )
    supplier_summary = supplier_summary.merge(
        iso_lookup,
        on="supplier_country",
        how="left"
    )
else:
    supplier_summary["country_iso3"] = np.nan

# Fill missing ISO3 codes using country_converter
missing_iso = supplier_summary["country_iso3"].isna()

supplier_summary.loc[missing_iso, "country_iso3"] = coco.convert(
    names=supplier_summary.loc[missing_iso, "supplier_country"].tolist(),
    to="ISO3",
    not_found=None
)

supplier_summary = supplier_summary.sort_values(
    "total_route_vulnerability_score",
    ascending=False
)

supplier_table_path = os.path.join(TABLE_DIR, f"italy_critical_import_suppliers_{YEAR}.csv")
supplier_summary.to_csv(supplier_table_path, index=False)
print("Saved supplier summary to:", supplier_table_path)

display(supplier_summary.head(20))


# ------------------------------------------------------------
# Preview and save map
# ------------------------------------------------------------

fig_map = px.choropleth(
    supplier_summary,
    locations="country_iso3",
    color="total_route_vulnerability_score",
    hover_name="supplier_country",
    hover_data={
        "country_iso3": False,
        "n_critical_routes": True,
        "critical_trade_value_musd": ":.2f",
        "avg_supplier_share": ":.2f",
        "max_supplier_share": ":.2f",
        "min_import_slack": ":.3f",
        "total_route_vulnerability_score": ":.2f"
    },
    title=f"Italy's Critical Import Supplier Countries, {YEAR}",
    labels={
        "total_route_vulnerability_score": "Route vulnerability score"
    }
)

fig_map.update_layout(
    margin=dict(l=0, r=0, t=60, b=0)
)

# Preview inside Colab
fig_map.show()

# Save interactive HTML
map_html_path = os.path.join(FIG_DIR, f"italy_critical_import_supplier_map_{YEAR}.html")
fig_map.write_html(map_html_path)
print("Saved interactive map to:", map_html_path)

# Save PNG and preview static image
map_png_path = os.path.join(FIG_DIR, f"italy_critical_import_supplier_map_{YEAR}.png")

try:
    fig_map.write_image(map_png_path, scale=2)
    print("Saved static map to:", map_png_path)
    display(Image(filename=map_png_path))
except Exception as e:
    print("Could not save PNG map. Interactive HTML was saved correctly.")
    print("Error:", e)


# ------------------------------------------------------------
# Preview and save bar chart of top supplier countries
# ------------------------------------------------------------

top_suppliers_bar = supplier_summary.head(15).copy()

fig_bar = px.bar(
    top_suppliers_bar.sort_values("total_route_vulnerability_score"),
    x="total_route_vulnerability_score",
    y="supplier_country",
    orientation="h",
    hover_data={
        "n_critical_routes": True,
        "critical_trade_value_musd": ":.2f",
        "avg_supplier_share": ":.2f",
        "min_import_slack": ":.3f"
    },
    title=f"Top Critical Import Supplier Countries for Italy, {YEAR}",
    labels={
        "total_route_vulnerability_score": "Total route vulnerability score",
        "supplier_country": "Supplier country"
    }
)

fig_bar.update_layout(
    margin=dict(l=120, r=20, t=60, b=40)
)

# Preview inside Colab
fig_bar.show()

# Save interactive HTML
bar_html_path = os.path.join(FIG_DIR, f"italy_top_critical_import_suppliers_{YEAR}.html")
fig_bar.write_html(bar_html_path)
print("Saved interactive bar chart to:", bar_html_path)

# Save PNG and preview static image
bar_png_path = os.path.join(FIG_DIR, f"italy_top_critical_import_suppliers_{YEAR}.png")

try:
    fig_bar.write_image(bar_png_path, scale=2)
    print("Saved static bar chart to:", bar_png_path)
    display(Image(filename=bar_png_path))
except Exception as e:
    print("Could not save PNG bar chart. Interactive HTML was saved correctly.")
    print("Error:", e)

### 14G. Poster visual: Italy’s critical import supplier countries

This cell builds a poster-oriented visualization of Italy’s import-side vulnerability. The question is: from which countries does Italy depend on critical imported products? A supplier-product route is defined as critical when \(Slack^{imp}_{cp}<1\), meaning that if supplier country \(c\) disappears, Italy’s remaining imports of product \(p\) would fall below the quota threshold.

The analysis keeps only critical import routes and computes a `route_vulnerability_score`, which combines three elements: the supplier’s share in Italy’s imports of the product, the depth of criticality, and the trade value of the route. This avoids ranking only by slack, because pure slack would overemphasize small single-supplier routes. The resulting score gives more visibility to routes that are both structurally fragile and economically relevant.

The output contains three components. First, the route-level table identifies the most vulnerable supplier-product routes for Italy. These include, among others, Norway for dredgers, India for iron and non-alloy steel ingots, Spain for nuclear reactors, the Netherlands for live plants, France for cattle, Türkiye for antimony ores, and the Russian Federation for iron pyrites. Second, the supplier-country summary aggregates these routes by country. Third, the map and bar chart visualize the geography of Italy’s critical import dependence.

The supplier-country ranking shows that Germany, China, France, Spain, and the Netherlands are the most relevant countries by total route vulnerability score. Germany has the largest number of critical routes, followed by China and France. This result should not be interpreted as direct geopolitical risk. Rather, it identifies structural import dependence: geopolitical vulnerability emerges when these critical supplier-product routes overlap with strategic sectors, political risk, logistical disruption, or low substitutability.

In [ ]:
# ============================================================
# 14H. SUPPLIER-COUNTRY NODE TABLE
# QUESTION:
#   "Which foreign supplier countries create the largest critical
#    import exposure for Italy?"
# ============================================================

def top_products_text(df, n=3):
    tmp = df.sort_values("route_vulnerability_score", ascending=False).head(n)

    labels = []
    for _, row in tmp.iterrows():
        desc = str(row["product_description"])
        if len(desc) > 45:
            desc = desc[:45] + "..."
        labels.append(
            f"{row['hs4']} ({row['hs2_section']}) - {desc}"
        )

    return " | ".join(labels)


critical_routes_by_supplier = (
    italy_critical_routes
    .groupby("supplier_country", as_index=False)
    .agg(
        n_critical_products=("hs4", "nunique"),
        n_critical_routes=("hs4", "size"),
        total_critical_imports=("weight", "sum"),
        total_critical_imports_musd=("trade_value_musd", "sum"),
        min_slack=("Slack_import_cp", "min"),
        avg_slack=("Slack_import_cp", "mean"),
        max_supplier_share=("supplier_share", "max"),
        avg_supplier_share=("supplier_share", "mean"),
        total_route_score=("route_vulnerability_score", "sum")
    )
)

top_products_by_supplier = (
    italy_critical_routes
    .groupby("supplier_country", group_keys=False)
    .apply(lambda df: pd.Series({
        "top_critical_products": top_products_text(df, n=3)
    }))
    .reset_index()
)

critical_routes_by_supplier = critical_routes_by_supplier.merge(
    top_products_by_supplier,
    on="supplier_country",
    how="left"
)

critical_routes_by_supplier = critical_routes_by_supplier.sort_values(
    ["total_route_score", "total_critical_imports_musd"],
    ascending=[False, False]
)

display(critical_routes_by_supplier.head(30))

### 14H. Supplier-country exposure table for Italy

This cell aggregates Italy’s critical import routes at the supplier-country level. While the previous cell visualizes the geography of Italy’s import vulnerability, this table provides a compact country-level summary suitable for the paper or poster.

For each foreign supplier country, the table reports the number of critical HS4 products, the total value of Italy’s critical imports from that country, the minimum and average import slack, the maximum and average supplier share, and the total route vulnerability score. The final column lists the main HS4 products driving each country’s critical exposure.

The results show that Germany, China, France, Spain, and the Netherlands generate the largest critical import exposure for Italy according to the total route vulnerability score. Germany ranks first, with 235 critical routes and approximately 24.98 billion USD in critical imports. China follows with 133 critical routes and approximately 12.83 billion USD, while France, Spain, and the Netherlands also appear as major sources of structurally critical import dependence.

This table answers the question: which foreign supplier countries generate the largest structural import exposure for Italy, and through which products? The ranking should not be interpreted as a direct geopolitical-risk ranking. A country may rank highly because Italy has many critical import routes from it, because those routes involve large trade values, or because the supplier dominates specific product markets. Geopolitical vulnerability emerges only when this structural dependence overlaps with strategic sectors, political risk, logistical disruption, or low substitutability.

The warning produced by `groupby.apply` is a pandas deprecation warning and does not affect the current results. It only indicates that future pandas versions will change how grouping columns are handled inside `apply`.

In [ ]:
# ============================================================
# 14I. WORLD MAP OF ITALY'S CRITICAL IMPORT ROUTES
# QUESTION:
#   "Which countries are connected to Italy through critical
#    import-dependence routes?"
#
# Visual encoding:
#   Node size  = number of critical products
#   Edge width = total critical import value
#   Edge color = severity based on minimum slack
# ============================================================

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import geopandas as gpd
import country_converter as coco
from IPython.display import Image, display

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def clean_country_label(name):
    """
    Fixes common encoding artifacts in country names for display.
    """
    replacements = {
        "TÃ¼rkiye": "Türkiye",
        "CÃ´te d'Ivoire": "Côte d'Ivoire",
        "CuraÃ§ao": "Curaçao"
    }
    return replacements.get(str(name), str(name))


def slack_color(slack):
    """
    Edge color based on the most severe import slack route.
    Lower slack = more severe dependence.
    """
    if slack < 0.50:
        return "#7f0000"   # severe
    elif slack < 1.00:
        return "#d7301f"   # critical
    elif slack < 1.10:
        return "#fc8d59"   # near-critical
    elif slack < 1.20:
        return "#fdbb84"   # watch
    else:
        return "#bdbdbd"   # redundant


def convert_to_iso3(country_series):
    """
    Converts country names to ISO3 codes.
    """
    converted = coco.convert(
        names=country_series.tolist(),
        to="ISO3",
        not_found=None
    )
    return converted


# ------------------------------------------------------------
# Load Natural Earth country polygons
# ------------------------------------------------------------

world_url = (
    "https://raw.githubusercontent.com/nvkelso/"
    "natural-earth-vector/master/geojson/"
    "ne_110m_admin_0_countries.geojson"
)

world = gpd.read_file(world_url)

# Standardize ISO column
if "ISO_A3" in world.columns:
    iso_col = "ISO_A3"
elif "ADM0_A3" in world.columns:
    iso_col = "ADM0_A3"
else:
    raise ValueError("Could not find ISO3 column in Natural Earth data.")

world = world[world[iso_col] != "-99"].copy()
world = world.rename(columns={iso_col: "iso3"})

# Compute centroids safely using projected CRS
world_projected = world.to_crs("EPSG:6933")
centroids_projected = world_projected.geometry.centroid

centroids = gpd.GeoSeries(
    centroids_projected,
    crs=world_projected.crs
).to_crs("EPSG:4326")

world["lon"] = centroids.x
world["lat"] = centroids.y


# ------------------------------------------------------------
# Prepare supplier map table
# ------------------------------------------------------------

supplier_map = critical_routes_by_supplier.copy()

# Clean labels for display
supplier_map["supplier_country_display"] = supplier_map["supplier_country"].apply(
    clean_country_label
)

# Convert supplier countries to ISO3
supplier_map["iso3"] = convert_to_iso3(supplier_map["supplier_country"])

# Manual ISO3 fixes if needed
manual_iso_fixes = {
    "TÃ¼rkiye": "TUR",
    "Türkiye": "TUR",
    "CÃ´te d'Ivoire": "CIV",
    "Côte d'Ivoire": "CIV",
    "Russian Federation": "RUS",
    "USA": "USA",
    "Rep. of Korea": "KOR",
    "Czechia": "CZE"
}

supplier_map["iso3"] = supplier_map.apply(
    lambda row: manual_iso_fixes.get(row["supplier_country"], row["iso3"]),
    axis=1
)

# Merge coordinates
supplier_map = supplier_map.merge(
    world[["iso3", "lon", "lat"]],
    on="iso3",
    how="left"
)

# Remove countries without coordinates
missing_coords = supplier_map[supplier_map["lon"].isna()][
    ["supplier_country", "iso3"]
]

if not missing_coords.empty:
    print("Countries without map coordinates:")
    display(missing_coords)

supplier_map = supplier_map.dropna(subset=["lon", "lat"]).copy()

# Italy coordinates
italy_iso3 = "ITA"
italy_row = world[world["iso3"] == italy_iso3].iloc[0]
italy_lon = italy_row["lon"]
italy_lat = italy_row["lat"]


# ------------------------------------------------------------
# Select top suppliers for readable poster map
# ------------------------------------------------------------

TOP_SUPPLIER_ROUTES = 10

supplier_map_plot = (
    supplier_map
    .sort_values("total_route_score", ascending=False)
    .head(TOP_SUPPLIER_ROUTES)
    .copy()
)

# Normalize edge widths by total critical import value
max_trade = supplier_map_plot["total_critical_imports_musd"].max()

supplier_map_plot["edge_width"] = (
    1.5
    + 7
    * supplier_map_plot["total_critical_imports_musd"]
    / max_trade
)

# Node size by number of critical products
supplier_map_plot["node_size"] = (
    0.3
    + 1 * np.sqrt(supplier_map_plot["n_critical_products"])
)


# ------------------------------------------------------------
# Build interactive route map
# ------------------------------------------------------------

fig_italy_import_routes_map = go.Figure()

# Edges: supplier country -> Italy
for _, row in supplier_map_plot.iterrows():

    hover_text = (
        f"<b>{row['supplier_country_display']} → Italy</b><br>"
        f"Critical products: {row['n_critical_products']}<br>"
        f"Critical routes: {row['n_critical_routes']}<br>"
        f"Critical imports: {row['total_critical_imports_musd']:.1f} million USD<br>"
        f"Total route score: {row['total_route_score']:.2f}<br>"
        f"Minimum slack: {row['min_slack']:.3f}<br>"
        f"Maximum supplier share: {row['max_supplier_share']:.2%}<br>"
        f"Top products: {row['top_critical_products']}"
    )

    fig_italy_import_routes_map.add_trace(
        go.Scattergeo(
            lon=[row["lon"], italy_lon],
            lat=[row["lat"], italy_lat],
            mode="lines",
            line=dict(
                width=row["edge_width"],
                color=slack_color(row["min_slack"])
            ),
            opacity=0.75,
            hoverinfo="text",
            text=hover_text,
            showlegend=False
        )
    )

# Supplier nodes
fig_italy_import_routes_map.add_trace(
    go.Scattergeo(
        lon=supplier_map_plot["lon"],
        lat=supplier_map_plot["lat"],
        mode="markers+text",
        text=supplier_map_plot["supplier_country_display"],
        textposition="top center",
        marker=dict(
            size=supplier_map_plot["node_size"],
            color=supplier_map_plot["total_route_score"],
            colorscale="Plasma",
            colorbar=dict(title="Route score"),
            line=dict(width=0.8, color="black")
        ),
        hovertext=[
            f"<b>{r.supplier_country_display}</b><br>"
            f"Critical products: {r.n_critical_products}<br>"
            f"Critical routes: {r.n_critical_routes}<br>"
            f"Critical imports: {r.total_critical_imports_musd:.1f} million USD<br>"
            f"Total route score: {r.total_route_score:.2f}<br>"
            f"Minimum slack: {r.min_slack:.3f}<br>"
            f"Top products: {r.top_critical_products}"
            for r in supplier_map_plot.itertuples()
        ],
        hoverinfo="text",
        name="Critical suppliers"
    )
)

# Italy node
fig_italy_import_routes_map.add_trace(
    go.Scattergeo(
        lon=[italy_lon],
        lat=[italy_lat],
        mode="markers+text",
        text=["Italy"],
        textposition="bottom center",
        marker=dict(
            size=20,
            color="royalblue",
            line=dict(width=1.5, color="black")
        ),
        hoverinfo="text",
        hovertext=["<b>Italy</b><br>Importing country"],
        name="Italy"
    )
)

fig_italy_import_routes_map.update_layout(
    title=(
        f"{QUESTION_IMPORT_ROUTES}<br>"
        f"<sup>Top {TOP_SUPPLIER_ROUTES} supplier countries, "
        f"critical routes only, year {YEAR}, theta={THETA}</sup>"
    ),
    geo=dict(
        projection_type="natural earth",
        showland=True,
        landcolor="rgb(245,245,245)",
        showcountries=True,
        countrycolor="rgb(180,180,180)",
        showocean=True,
        oceancolor="rgb(235,245,255)",
        showframe=False
    ),
    width=1200,
    height=750,
    margin=dict(l=20, r=20, t=80, b=20)
)

# Preview interactive map inside Colab
fig_italy_import_routes_map.show()


# ------------------------------------------------------------
# Save map as HTML and PNG
# ------------------------------------------------------------

map_html_path = os.path.join(
    FIG_DIR,
    f"italy_critical_import_routes_map_{YEAR}.html"
)

map_png_path = os.path.join(
    FIG_DIR,
    f"italy_critical_import_routes_map_{YEAR}.png"
)

fig_italy_import_routes_map.write_html(map_html_path)
print("Saved HTML:", map_html_path)

try:
    fig_italy_import_routes_map.write_image(map_png_path, scale=2)
    print("Saved PNG:", map_png_path)
    display(Image(filename=map_png_path))
except Exception as e:
    print("PNG export failed. HTML was saved correctly.")
    print(e)


# ------------------------------------------------------------
# Display plotted supplier table for verification
# ------------------------------------------------------------

display(
    supplier_map_plot[
        [
            "supplier_country_display",
            "n_critical_products",
            "n_critical_routes",
            "total_critical_imports_musd",
            "total_route_score",
            "min_slack",
            "max_supplier_share",
            "top_critical_products"
        ]
    ]
)

### 14I. World map of Italy’s critical import routes

This cell creates a route-based world map of Italy’s critical import dependencies. While the previous visualizations aggregate vulnerability by supplier country, this map represents the main foreign supplier countries as nodes connected to Italy. Each route summarizes the critical import exposure between a supplier country and Italy.

The visual encoding is designed for poster interpretation. Node size reflects the number of critical HS4 products supplied by each country. Edge width reflects the total value of Italy’s critical imports from that country. Edge color is based on the minimum import slack observed among that supplier’s critical routes, so darker edges indicate more severe dependence.

This map should be interpreted as a geographic visualization of structural import dependence, not as a geopolitical-risk map. A route is critical when \(Slack^{imp}_{cp}<1\), meaning that Italy would fall below the quota threshold for at least one imported product if that supplier were removed. Geopolitical vulnerability would require an additional layer of information, such as political risk, strategic-sector classification, transport constraints, or substitutability.

The map focuses on the top supplier countries for readability. It is therefore best used as a poster visual or exploratory figure, while the supplier-country table and bar chart provide the more precise quantitative ranking.

In [ ]:
# ============================================================
# 14J. PRODUCT-SPECIFIC CRITICAL ROUTE MAP
# QUESTION:
#   "For a specific vulnerable product, which countries supply Italy,
#    and which supplier routes are critical?"
# ============================================================

from IPython.display import Image, display

QUESTION_PRODUCT_ROUTE = (
    "For a specific vulnerable product, which supplier routes to Italy are critical?"
)


def plot_product_specific_italy_map(
    imports_df,
    product_code=2002,
    top_suppliers=10
):
    """
    Creates a product-specific import route map for Italy.

    For a selected HS4 product, the function shows:
      - which countries supply Italy,
      - each supplier's share in Italy's imports of that product,
      - whether each supplier route is critical according to Slack_import_cp < 1.

    Interpretation:
      A route is critical if removing that supplier would leave Italy
      below the quota threshold for that product.
    """

    df = imports_df.copy()
    df["hs4"] = df["hs4"].astype(str).str.zfill(4)

    # If no product is selected, choose the product with the highest route vulnerability score
    if product_code is None:
        product_code = italy_critical_route_table.iloc[0]["hs4"]

    product_code = str(product_code).zfill(4)

    product_df = df[df["hs4"] == product_code].copy()

    if product_df.empty:
        raise ValueError(f"No records found for product {product_code}.")

    product_df["trade_value_musd"] = product_df["weight"] / 1000

    product_df = product_df.sort_values(
        ["supplier_share", "weight"],
        ascending=[False, False]
    ).head(top_suppliers)

    product_description = product_df["product_description"].iloc[0]

    # Clean country labels for display
    product_df["supplier_country_display"] = product_df["supplier_country"].apply(
        clean_country_label
    )

    # Convert supplier country names to ISO3
    product_df["iso3"] = convert_to_iso3(product_df["supplier_country"])

    # Manual ISO3 fixes for common problematic labels
    manual_iso_fixes = {
        "TÃ¼rkiye": "TUR",
        "Türkiye": "TUR",
        "CÃ´te d'Ivoire": "CIV",
        "Côte d'Ivoire": "CIV",
        "Russian Federation": "RUS",
        "USA": "USA",
        "Rep. of Korea": "KOR",
        "Czechia": "CZE",
        "France": "FRA",
        "Norway": "NOR",
        "Singapore": "SGP",
        "China, Hong Kong SAR": "HKG",
        "Other Asia, nes": "TWN"
    }

    product_df["iso3"] = product_df.apply(
        lambda row: manual_iso_fixes.get(row["supplier_country"], row["iso3"]),
        axis=1
    )

    # Merge supplier coordinates
    product_df = product_df.merge(
        world[["iso3", "lon", "lat"]],
        on="iso3",
        how="left"
    )

    missing_coords = product_df[product_df["lon"].isna()][
        ["supplier_country", "iso3"]
    ]

    if not missing_coords.empty:
        print("Suppliers without map coordinates:")
        display(missing_coords)

    product_df = product_df.dropna(subset=["lon", "lat"]).copy()

    if product_df.empty:
        raise ValueError(
            f"No supplier coordinates available for product {product_code}."
        )

    max_trade = product_df["trade_value_musd"].max()

    product_df["edge_width"] = (
        1.5 + 7 * product_df["trade_value_musd"] / max_trade
    )

    product_df["node_size"] = (
        10 + 30 * product_df["supplier_share"]
    )

    # ------------------------------------------------------------
    # Build map
    # ------------------------------------------------------------

    fig = go.Figure()

    # Supplier-to-Italy edges
    for _, row in product_df.iterrows():

        is_critical = row["Slack_import_cp"] < 1

        hover_text = (
            f"<b>{row['supplier_country_display']} → Italy</b><br>"
            f"HS4: {product_code}<br>"
            f"{product_description}<br>"
            f"Trade value: {row['trade_value_musd']:.1f} million USD<br>"
            f"Supplier share in Italy's imports: {row['supplier_share']:.2%}<br>"
            f"Slack: {row['Slack_import_cp']:.3f}<br>"
            f"Status: {row['risk_label']}"
        )

        fig.add_trace(
            go.Scattergeo(
                lon=[row["lon"], italy_lon],
                lat=[row["lat"], italy_lat],
                mode="lines",
                line=dict(
                    width=row["edge_width"],
                    color=slack_color(row["Slack_import_cp"])
                    if is_critical else "#bdbdbd"
                ),
                opacity=0.75,
                hoverinfo="text",
                text=hover_text,
                showlegend=False
            )
        )

    # Supplier nodes
    fig.add_trace(
        go.Scattergeo(
            lon=product_df["lon"],
            lat=product_df["lat"],
            mode="markers+text",
            text=product_df["supplier_country_display"],
            textposition="top center",
            marker=dict(
                size=product_df["node_size"],
                color=product_df["Slack_import_cp"],
                colorscale="Reds_r",
                colorbar=dict(title="Slack"),
                line=dict(width=0.8, color="black")
            ),
            hovertext=[
                f"<b>{r.supplier_country_display}</b><br>"
                f"Share: {r.supplier_share:.2%}<br>"
                f"Trade value: {r.trade_value_musd:.1f} million USD<br>"
                f"Slack: {r.Slack_import_cp:.3f}<br>"
                f"Status: {r.risk_label}"
                for r in product_df.itertuples()
            ],
            hoverinfo="text",
            name="Suppliers"
        )
    )

    # Italy node
    fig.add_trace(
        go.Scattergeo(
            lon=[italy_lon],
            lat=[italy_lat],
            mode="markers+text",
            text=["Italy"],
            textposition="bottom center",
            marker=dict(
                size=20,
                color="royalblue",
                line=dict(width=1.5, color="black")
            ),
            hoverinfo="text",
            hovertext=["<b>Italy</b><br>Importing country"],
            name="Italy"
        )
    )

    fig.update_layout(
        title=(
            f"{QUESTION_PRODUCT_ROUTE}<br>"
            f"<sup>Product {product_code}: {str(product_description)[:100]}, "
            f"year {YEAR}, theta={THETA}</sup>"
        ),
        geo=dict(
            projection_type="natural earth",
            showland=True,
            landcolor="rgb(245,245,245)",
            showcountries=True,
            countrycolor="rgb(180,180,180)",
            showocean=True,
            oceancolor="rgb(235,245,255)",
            showframe=False
        ),
        width=1200,
        height=750,
        margin=dict(l=20, r=20, t=90, b=20)
    )

    # Preview interactive map
    fig.show()

    # ------------------------------------------------------------
    # Save outputs
    # ------------------------------------------------------------

    html_path = os.path.join(
        FIG_DIR,
        f"italy_hs4_{product_code}_supplier_routes_{YEAR}.html"
    )

    png_path = os.path.join(
        FIG_DIR,
        f"italy_hs4_{product_code}_supplier_routes_{YEAR}.png"
    )

    fig.write_html(html_path)
    print("Saved HTML:", html_path)

    try:
        fig.write_image(png_path, scale=2)
        print("Saved PNG:", png_path)
        display(Image(filename=png_path))
    except Exception as e:
        print("PNG export failed. HTML was saved correctly.")
        print(e)

    # ------------------------------------------------------------
    # Return table used in the map
    # ------------------------------------------------------------

    product_df = product_df.sort_values(
        ["supplier_share", "weight"],
        ascending=[False, False]
    )

    return product_df


# Example: visualize Italy's supplier structure for HS4 2002
# Set product_code=None to automatically select the top route
# from italy_critical_route_table.
italy_product_specific_map_data = plot_product_specific_italy_map(
    italy_import_vulnerability,
    product_code=2002,
    top_suppliers=15
)

display(
    italy_product_specific_map_data[
        [
            "supplier_country_display",
            "supplier_country",
            "hs4",
            "product_description",
            "trade_value_musd",
            "supplier_share",
            "Slack_import_cp",
            "risk_label"
        ]
    ]
)

### 14J. Product-specific critical import route map

This cell creates a product-specific map of Italy’s import suppliers for one selected HS4 product. While the previous map summarizes Italy’s critical import dependence across supplier countries, this visualization focuses on a single product and shows which countries supply Italy, how large each supplier’s share is, and whether each route is critical according to the import slack criterion.

For a selected HS4 product \(p\), the map displays supplier countries connected to Italy. Edge width is proportional to the trade value of the route, while node size reflects the supplier’s share in Italy’s imports of that product. Critical routes are those for which \(Slack^{imp}_{cp}<1\), meaning that if the supplier disappeared, Italy’s remaining imports of that product would fall below the quota threshold.

This figure is useful as a case-study visualization. For example, using HS4 `2002` focuses on tomato preparations, where the map can show whether Italy’s imports are concentrated in one or a few suppliers or whether there are multiple relevant alternatives. This product-level view helps translate the slack metric into an intuitive supply-chain interpretation.

The cell should be interpreted as an importer-side analysis: Italy is the importing country, and the foreign countries are suppliers. It is not measuring Italy’s export-side pivotality. The visualization is best used for the poster or as an appendix figure, while the aggregate supplier-country table remains more suitable for the main paper.

In [ ]:
# ============================================================
# 14K. COUNTRY-NODE NETWORK GRAPH
# QUESTION:
#   "Which supplier countries form Italy's critical import-dependence network?"
#
# Nodes:
#   Countries
#
# Edges:
#   Critical supplier -> Italy routes
#
# Edge width:
#   Total critical import value
#
# Node size:
#   Number of critical products
# ============================================================

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import os

QUESTION_COUNTRY_NETWORK = (
    "Which supplier countries form Italy's critical import-dependence network?"
)

print(QUESTION_COUNTRY_NETWORK)

# ------------------------------------------------------------
# Select top supplier countries directly from the supplier table
# ------------------------------------------------------------

TOP_SUPPLIER_ROUTES = 10

network_data = (
    critical_routes_by_supplier
    .sort_values("total_route_score", ascending=False)
    .head(TOP_SUPPLIER_ROUTES)
    .copy()
)

# Clean labels for display
network_data["supplier_country_display"] = network_data["supplier_country"].apply(
    clean_country_label
)

# ------------------------------------------------------------
# Build graph
# ------------------------------------------------------------

G = nx.Graph()
G.add_node("Italy", node_type="importer")

for _, row in network_data.iterrows():

    supplier = row["supplier_country_display"]

    G.add_node(
        supplier,
        node_type="supplier",
        n_critical_products=row["n_critical_products"],
        n_critical_routes=row["n_critical_routes"],
        min_slack=row["min_slack"],
        total_critical_imports_musd=row["total_critical_imports_musd"],
        total_route_score=row["total_route_score"]
    )

    G.add_edge(
        supplier,
        "Italy",
        weight=row["total_critical_imports_musd"],
        min_slack=row["min_slack"],
        total_route_score=row["total_route_score"],
        n_critical_products=row["n_critical_products"],
        top_critical_products=row["top_critical_products"]
    )

# ------------------------------------------------------------
# Circular layout: Italy in center, suppliers around
# ------------------------------------------------------------

pos = {"Italy": np.array([0.0, 0.0])}

suppliers = [n for n in G.nodes() if n != "Italy"]
angles = np.linspace(0, 2 * np.pi, len(suppliers), endpoint=False)

for node, angle in zip(suppliers, angles):
    pos[node] = np.array([np.cos(angle), np.sin(angle)])

# ------------------------------------------------------------
# Node sizes and colors
# ------------------------------------------------------------

node_sizes = []
node_colors = []

for node in G.nodes():
    if node == "Italy":
        node_sizes.append(2200)
        node_colors.append("royalblue")
    else:
        ncrit = G.nodes[node]["n_critical_products"]

        # Square-root scaling prevents Germany/China from becoming unreadably large
        node_sizes.append(600 + 350 * np.sqrt(ncrit))
        node_colors.append("lightcoral")

# ------------------------------------------------------------
# Edge widths and colors
# ------------------------------------------------------------

edge_widths = []
edge_colors = []

max_edge_weight = max([G[u][v]["weight"] for u, v in G.edges()])

for u, v in G.edges():
    edge_widths.append(
        1.0 + 7.0 * G[u][v]["weight"] / max_edge_weight
    )
    edge_colors.append(slack_color(G[u][v]["min_slack"]))

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(12, 10))

nx.draw_networkx_edges(
    G,
    pos,
    width=edge_widths,
    edge_color=edge_colors,
    alpha=0.75
)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="black",
    linewidths=1.0
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=9
)

plt.title(
    f"{QUESTION_COUNTRY_NETWORK}\n"
    f"Top {TOP_SUPPLIER_ROUTES} supplier countries, year {YEAR}, theta={THETA}"
)

plt.axis("off")
plt.tight_layout()

network_png_path = os.path.join(
    FIG_DIR,
    f"italy_country_node_critical_import_network_{YEAR}.png"
)

plt.savefig(network_png_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved network graph:", network_png_path)

# ------------------------------------------------------------
# Display data used in the network
# ------------------------------------------------------------

display(
    network_data[
        [
            "supplier_country_display",
            "n_critical_products",
            "n_critical_routes",
            "total_critical_imports_musd",
            "total_route_score",
            "min_slack",
            "max_supplier_share",
            "top_critical_products"
        ]
    ]
)

### 14K. Country-node network of Italy’s critical import dependence

This cell creates a country-level network graph of Italy’s critical import-dependence structure. Italy is represented as the central importing country, while foreign supplier countries are represented as surrounding nodes. An edge between a supplier and Italy summarizes all critical supplier-product routes from that country to Italy.

The graph is designed as a compact poster visualization. Node size reflects the number of critical HS4 products supplied by each country, while edge width reflects the total value of critical imports. Edge color is based on the minimum import slack observed for each supplier country, so darker edges indicate more severe dependence.

This visualization is complementary to the map and supplier-country table. The map emphasizes geography, the table provides exact values, and this network graph emphasizes the structure of Italy’s critical supplier dependence. It should be interpreted as a structural-dependence network, not as a geopolitical-risk network. Countries such as Germany, China, France, Spain, and the Netherlands may appear central because Italy has many critical import routes from them, because those routes involve large values, or because some products have very low import slack.

In [ ]:
# ============================================================
# 14L. SANKEY DIAGRAM
# QUESTION:
#   "Through which product sectors do critical import routes connect
#    foreign suppliers to Italy?"
#
# Structure:
#   Supplier country -> HS2 macro-section -> Italy
# ============================================================

from IPython.display import Image, display

QUESTION_SANKEY = (
    "Through which product sectors do critical import routes connect foreign suppliers to Italy?"
)

print(QUESTION_SANKEY)

TOP_SUPPLIERS_SANKEY = 12

# ------------------------------------------------------------
# Select top supplier countries by total route score
# ------------------------------------------------------------

top_supplier_names = (
    critical_routes_by_supplier
    .sort_values("total_route_score", ascending=False)
    .head(TOP_SUPPLIERS_SANKEY)["supplier_country"]
    .tolist()
)

sankey_df = italy_critical_routes[
    italy_critical_routes["supplier_country"].isin(top_supplier_names)
].copy()

# Clean display labels
sankey_df["supplier_country_display"] = sankey_df["supplier_country"].apply(
    clean_country_label
)

# ------------------------------------------------------------
# Aggregate supplier -> sector
# ------------------------------------------------------------

supplier_sector = (
    sankey_df
    .groupby(["supplier_country_display", "hs2_section"], as_index=False)
    .agg(
        value=("trade_value_musd", "sum"),
        n_products=("hs4", "nunique"),
        min_slack=("Slack_import_cp", "min")
    )
)

# Aggregate sector -> Italy
sector_italy = (
    supplier_sector
    .groupby("hs2_section", as_index=False)
    .agg(value=("value", "sum"))
)

# ------------------------------------------------------------
# Build Sankey nodes
# ------------------------------------------------------------

suppliers = sorted(supplier_sector["supplier_country_display"].unique())
sectors = sorted(supplier_sector["hs2_section"].unique())
final_node = ["Italy"]

nodes = suppliers + sectors + final_node
node_index = {node: idx for idx, node in enumerate(nodes)}

sources = []
targets = []
values = []
hover_labels = []

# Supplier -> sector links
for _, row in supplier_sector.iterrows():
    sources.append(node_index[row["supplier_country_display"]])
    targets.append(node_index[row["hs2_section"]])
    values.append(row["value"])
    hover_labels.append(
        f"{row['supplier_country_display']} → {row['hs2_section']}<br>"
        f"Critical imports: {row['value']:.1f} million USD<br>"
        f"Critical products: {row['n_products']}<br>"
        f"Minimum slack: {row['min_slack']:.3f}"
    )

# Sector -> Italy links
for _, row in sector_italy.iterrows():
    sources.append(node_index[row["hs2_section"]])
    targets.append(node_index["Italy"])
    values.append(row["value"])
    hover_labels.append(
        f"{row['hs2_section']} → Italy<br>"
        f"Critical imports: {row['value']:.1f} million USD"
    )

# ------------------------------------------------------------
# Node colors
# ------------------------------------------------------------

supplier_color = "rgba(220, 20, 60, 0.75)"      # crimson
sector_color   = "rgba(255, 165, 0, 0.75)"      # orange
italy_color    = "rgba(65, 105, 225, 0.9)"      # royalblue

node_colors = (
    [supplier_color] * len(suppliers)
    + [sector_color] * len(sectors)
    + [italy_color]
)

# Optional uniform link color
link_colors = ["rgba(160,160,160,0.45)"] * len(values)

# ------------------------------------------------------------
# Create Sankey figure
# ------------------------------------------------------------

fig_italy_sankey = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=18,
                line=dict(color="black", width=0.5),
                label=nodes,
                color=node_colors
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color=link_colors,
                customdata=hover_labels,
                hovertemplate="%{customdata}<extra></extra>"
            )
        )
    ]
)

fig_italy_sankey.update_layout(
    title=(
        f"{QUESTION_SANKEY}<br>"
        f"<sup>Top {TOP_SUPPLIERS_SANKEY} supplier countries, "
        f"critical routes only, year {YEAR}, theta={THETA}</sup>"
    ),
    width=1200,
    height=750,
    margin=dict(l=20, r=20, t=80, b=20)
)

# Preview interactive figure
fig_italy_sankey.show()

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

sankey_html_path = os.path.join(
    FIG_DIR,
    f"italy_critical_import_routes_sankey_{YEAR}.html"
)

sankey_png_path = os.path.join(
    FIG_DIR,
    f"italy_critical_import_routes_sankey_{YEAR}.png"
)

fig_italy_sankey.write_html(sankey_html_path)

try:
    fig_italy_sankey.write_image(sankey_png_path, scale=2)
    print("Saved PNG:", sankey_png_path)
    display(Image(filename=sankey_png_path))
except Exception as e:
    print("PNG export failed. HTML was saved correctly.")
    print(e)

print("Saved HTML:", sankey_html_path)

# ------------------------------------------------------------
# Preview aggregated tables
# ------------------------------------------------------------

print("\nSupplier -> sector flows:")
display(
    supplier_sector.sort_values("value", ascending=False).head(20)
)

print("\nSector -> Italy flows:")
display(
    sector_italy.sort_values("value", ascending=False)
)

### 14L. Sankey diagram of Italy’s critical import dependence by sector

This cell creates a Sankey diagram to summarize how Italy’s critical import routes are distributed across supplier countries and broad product sectors. The structure of the diagram is:

\[
\text{Supplier country} \rightarrow \text{HS2 macro-section} \rightarrow \text{Italy}.
\]

The diagram is built using only critical import routes, that is, supplier-product links for which \(Slack^{imp}_{cp}<1\). The width of each flow is proportional to the value of critical imports, expressed in million USD. This allows the visualization to show not only which countries matter, but also through which product sectors Italy’s structural import dependence is concentrated.

The results show that the largest critical flows are concentrated in machinery and electronics, chemicals, other manufactured goods, minerals and fuels, stone/glass/precious metals, and base metals. The largest sectoral flow is machinery and electronics, with approximately 21.7 billion USD in critical imports, followed by chemicals with approximately 13.4 billion USD and other manufactured goods with approximately 8.4 billion USD. These sectors are especially relevant for the paper because they connect the slack-based measure to strategic autonomy, industrial supply chains, and trade fragmentation.

At the supplier-sector level, Germany and China dominate critical flows in machinery and electronics, while Germany also contributes strongly through base metals, plastics and rubber, other manufactured goods, and chemicals. France appears in minerals and fuels, chemicals, animals, and stone/glass/precious metals. Switzerland is relevant in minerals and fuels, stone/glass/precious metals, and chemicals, while the Netherlands appears prominently in other manufactured goods and machinery and electronics.

This figure complements the supplier-country table and the route-level rankings. The supplier-country table identifies who creates Italy’s largest critical import exposure, while the Sankey diagram shows through which sectors this exposure reaches Italy. The diagram should be interpreted as a descriptive visualization of structural dependence, not as a direct geopolitical-risk measure. Geopolitical vulnerability emerges when these critical flows overlap with strategic sectors, political risk, transport bottlenecks, or limited substitutability.

Note: the interactive HTML version was saved correctly. The PNG export failed because the `kaleido` package was not available or not correctly loaded in the current environment. This does not affect the results or the interactive visualization. To export a static PNG, install or update Kaleido and rerun the cell.

In [ ]:
# ============================================================
# 15. SAVE IMPORTANT OUTPUTS FOR POSTER / PAPER
# ============================================================

import os
import pickle
from datetime import datetime

# ------------------------------------------------------------
# 15.1 Output folders
# ------------------------------------------------------------

OUT_DIR = os.path.join(DATA_DIR, "results_country_product_weighted_slack")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TABLE_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print("Output folder:", OUT_DIR)
print("Tables folder:", TABLE_DIR)
print("Figures folder:", FIG_DIR)


# ------------------------------------------------------------
# 15.2 Helper functions
# ------------------------------------------------------------

def save_csv_if_exists(var_name, filename):
    """
    Save a dataframe to CSV only if it exists in the notebook.
    """
    if var_name in globals():
        obj = globals()[var_name]
        if isinstance(obj, pd.DataFrame):
            path = os.path.join(TABLE_DIR, filename)
            obj.to_csv(path, index=False)
            print(f"Saved CSV: {filename}")
        else:
            print(f"Skipped {var_name}: object exists but is not a DataFrame.")
    else:
        print(f"Skipped {var_name}: object not found.")


def save_excel_sheet_if_exists(writer, var_name, sheet_name, n=None, sort_col=None, ascending=True):
    """
    Save a dataframe to an Excel sheet if it exists.
    Optionally sort and keep only first n rows.
    """
    if var_name not in globals():
        print(f"Skipped Excel sheet {sheet_name}: {var_name} not found.")
        return

    df = globals()[var_name]

    if not isinstance(df, pd.DataFrame):
        print(f"Skipped Excel sheet {sheet_name}: {var_name} is not a DataFrame.")
        return

    df_out = df.copy()

    if sort_col is not None and sort_col in df_out.columns:
        df_out = df_out.sort_values(sort_col, ascending=ascending)

    if n is not None:
        df_out = df_out.head(n)

    df_out.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved Excel sheet: {sheet_name[:31]}")


def save_fig(path):
    """
    Save current matplotlib figure.
    """
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {os.path.basename(path)}")


# ------------------------------------------------------------
# 15.3 Save full CSV tables
# ------------------------------------------------------------

tables_to_save = {
    "edges_cp": f"edges_country_product_slack_{YEAR}.csv",
    "country_slack": f"country_weighted_slack_{YEAR}.csv",
    "product_slack": f"product_fragility_summary_{YEAR}.csv",
    "critical_edges": f"critical_country_product_edges_{YEAR}.csv",
    "criticality_contribution_edges": f"criticality_contribution_edges_{YEAR}.csv",
    "top_pivotal_countries_by_slack": f"top_pivotal_countries_by_weighted_slack_{YEAR}.csv",
    "top_fragile_products": f"top_fragile_products_{YEAR}.csv",
    "systemic_fragile_products": f"systemic_fragile_products_{YEAR}.csv",
    "hs2_fragility_summary": f"hs2_fragility_summary_{YEAR}.csv",
    "italy_critical_products": f"italy_critical_products_exporter_side_{YEAR}.csv",
    "italy_import_vulnerability": f"italy_import_vulnerability_importer_side_{YEAR}.csv",
    "italy_critical_routes": f"italy_critical_import_routes_full_{YEAR}.csv",
    "italy_critical_route_table": f"italy_critical_import_route_table_{YEAR}.csv",
    "critical_routes_by_supplier": f"italy_critical_routes_by_supplier_{YEAR}.csv",
    "supplier_map_plot": f"italy_supplier_map_plot_data_{YEAR}.csv",
    "italy_product_specific_map_data": f"italy_product_specific_map_data_{YEAR}.csv",
    "italy_bipartite_subgraph": f"italy_bipartite_subgraph_edges_{YEAR}.csv",
    "supplier_sector": f"italy_sankey_supplier_sector_flows_{YEAR}.csv",
    "sector_italy": f"italy_sankey_sector_flows_{YEAR}.csv"
}

for var_name, filename in tables_to_save.items():
    save_csv_if_exists(var_name, filename)


# ------------------------------------------------------------
# 15.4 Save compact Excel workbook
# ------------------------------------------------------------

excel_path = os.path.join(
    OUT_DIR,
    f"slack_analysis_summary_tables_{YEAR}.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    save_excel_sheet_if_exists(
        writer,
        "country_slack",
        "countries_weighted_slack",
        n=200,
        sort_col="Slack_weighted",
        ascending=True
    )

    save_excel_sheet_if_exists(
        writer,
        "product_slack",
        "products_fragility",
        n=500,
        sort_col="product_fragility_score",
        ascending=False
    )

    save_excel_sheet_if_exists(
        writer,
        "critical_edges",
        "critical_edges",
        n=500,
        sort_col="Slack_cp",
        ascending=True
    )

    save_excel_sheet_if_exists(
        writer,
        "top_fragile_products",
        "top_fragile_products",
        n=300,
        sort_col="product_fragility_score",
        ascending=False
    )

    save_excel_sheet_if_exists(
        writer,
        "systemic_fragile_products",
        "systemic_products",
        n=300,
        sort_col="systemic_fragility_rank",
        ascending=True
    )

    save_excel_sheet_if_exists(
        writer,
        "hs2_fragility_summary",
        "hs2_summary",
        n=None,
        sort_col="combined_hs2_fragility_rank",
        ascending=True
    )

    save_excel_sheet_if_exists(
        writer,
        "italy_critical_products",
        "italy_exports",
        n=200,
        sort_col="country_criticality_contribution",
        ascending=False
    )

    save_excel_sheet_if_exists(
        writer,
        "italy_import_vulnerability",
        "italy_imports",
        n=500,
        sort_col="Slack_import_cp",
        ascending=True
    )

    save_excel_sheet_if_exists(
        writer,
        "italy_critical_route_table",
        "italy_critical_routes",
        n=500,
        sort_col="route_vulnerability_score",
        ascending=False
    )

    save_excel_sheet_if_exists(
        writer,
        "critical_routes_by_supplier",
        "italy_supplier_exposure",
        n=200,
        sort_col="total_route_score",
        ascending=False
    )

print("Saved Excel workbook:", excel_path)


# ------------------------------------------------------------
# 15.5 Save figures: distribution of country weighted slack
# ------------------------------------------------------------

if "country_slack" in globals():
    plt.figure(figsize=(8, 5))
    plt.hist(country_slack["Slack_weighted"].dropna(), bins=40)
    plt.axvline(1, linestyle="--")
    plt.title(f"Distribution of Country Weighted Slack, {YEAR}")
    plt.xlabel("Country weighted slack")
    plt.ylabel("Number of countries")
    save_fig(os.path.join(FIG_DIR, f"distribution_country_weighted_slack_{YEAR}.png"))


# ------------------------------------------------------------
# 15.6 Save figures: distribution of product fragility score
# ------------------------------------------------------------

if "product_slack" in globals() and "product_fragility_score" in product_slack.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(product_slack["product_fragility_score"].dropna(), bins=40)
    plt.title(f"Distribution of Product Fragility Score, {YEAR}")
    plt.xlabel("Product fragility score")
    plt.ylabel("Number of products")
    save_fig(os.path.join(FIG_DIR, f"distribution_product_fragility_score_{YEAR}.png"))


# ------------------------------------------------------------
# 15.7 Save figures: top pivotal countries by weighted slack
# ------------------------------------------------------------

if "top_pivotal_countries_by_slack" in globals():
    top_countries_plot = top_pivotal_countries_by_slack.head(20).copy()

    plt.figure(figsize=(10, 6))
    plt.barh(
        top_countries_plot["country_name"][::-1],
        top_countries_plot["Slack_weighted"][::-1]
    )
    plt.title(f"Top Pivotal Countries by Portfolio Weighted Slack, {YEAR}")
    plt.xlabel("Weighted slack: lower = more pivotal")
    plt.ylabel("Country")
    save_fig(os.path.join(FIG_DIR, f"top_pivotal_countries_weighted_slack_{YEAR}.png"))


# ------------------------------------------------------------
# 15.8 Save figures: top fragile products
# ------------------------------------------------------------

if "top_fragile_products" in globals():
    top_products_plot = top_fragile_products.head(20).copy()

    labels = (
        top_products_plot["hs4"].astype(str)
        + " - "
        + top_products_plot["product_description"].astype(str).str.slice(0, 45)
    )

    plt.figure(figsize=(11, 7))
    plt.barh(
        labels[::-1],
        top_products_plot["product_fragility_score"][::-1]
    )
    plt.title(f"Most Fragile Products by Product Fragility Score, {YEAR}")
    plt.xlabel("Product fragility score")
    plt.ylabel("Product")
    save_fig(os.path.join(FIG_DIR, f"top_fragile_products_fragility_score_{YEAR}.png"))


# ------------------------------------------------------------
# 15.9 Save figures: dominant supplier share distribution
# ------------------------------------------------------------

if "product_slack" in globals() and "max_supply_share" in product_slack.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(product_slack["max_supply_share"].dropna(), bins=40)
    plt.axvline(1 - THETA, linestyle="--")
    plt.axvline(0.50, linestyle="--")
    plt.title(f"Distribution of Dominant Supplier Share by Product, {YEAR}")
    plt.xlabel("Maximum supplier share")
    plt.ylabel("Number of products")
    save_fig(os.path.join(FIG_DIR, f"distribution_top_supplier_share_{YEAR}.png"))


# ------------------------------------------------------------
# 15.10 Save figures: HHI vs product fragility score
# ------------------------------------------------------------

if "product_slack" in globals() and {"HHI", "product_fragility_score"}.issubset(product_slack.columns):
    plt.figure(figsize=(7, 5))
    plt.scatter(
        product_slack["HHI"],
        product_slack["product_fragility_score"],
        alpha=0.5
    )
    plt.title(f"HHI vs Threshold-Sensitive Product Fragility, {YEAR}")
    plt.xlabel("HHI")
    plt.ylabel("Product fragility score")
    save_fig(os.path.join(FIG_DIR, f"hhi_vs_product_fragility_score_{YEAR}.png"))


# ------------------------------------------------------------
# 15.11 Save figures: systemic product fragility
# ------------------------------------------------------------

if "systemic_fragile_products" in globals():
    systemic_plot = systemic_fragile_products.head(20).copy()

    labels = (
        systemic_plot["hs4"].astype(str)
        + " - "
        + systemic_plot["product_description"].astype(str).str.slice(0, 45)
    )

    plt.figure(figsize=(11, 7))
    plt.barh(
        labels[::-1],
        systemic_plot["product_fragility_score"][::-1]
    )
    plt.title(f"Systemically Relevant Fragile Products, {YEAR}")
    plt.xlabel("Product fragility score among top systemic products")
    plt.ylabel("Product")
    save_fig(os.path.join(FIG_DIR, f"systemic_fragile_products_{YEAR}.png"))


# ------------------------------------------------------------
# 15.12 Save figures: Italy critical export products
# ------------------------------------------------------------

if "italy_critical_products" in globals() and isinstance(italy_critical_products, pd.DataFrame) and not italy_critical_products.empty:
    italy_export_plot = italy_critical_products.head(20).copy()

    labels = (
        italy_export_plot["hs4"].astype(str)
        + " - "
        + italy_export_plot["product_description"].astype(str).str.slice(0, 45)
    )

    plt.figure(figsize=(11, 7))
    plt.barh(
        labels[::-1],
        italy_export_plot["country_criticality_contribution"][::-1]
    )
    plt.title(f"Italy: Critical Export Products by Contribution, {YEAR}")
    plt.xlabel("Country criticality contribution")
    plt.ylabel("Product")
    save_fig(os.path.join(FIG_DIR, f"italy_critical_export_products_{YEAR}.png"))


# ------------------------------------------------------------
# 15.13 Save figures: Italy import vulnerability by route score
# ------------------------------------------------------------

if "italy_critical_route_table" in globals() and isinstance(italy_critical_route_table, pd.DataFrame) and not italy_critical_route_table.empty:
    italy_import_plot = italy_critical_route_table.head(20).copy()

    labels = (
        italy_import_plot["supplier_country"].astype(str)
        + " | "
        + italy_import_plot["hs4"].astype(str)
        + " - "
        + italy_import_plot["product_description"].astype(str).str.slice(0, 40)
    )

    plt.figure(figsize=(12, 8))
    plt.barh(
        labels[::-1],
        italy_import_plot["route_vulnerability_score"][::-1]
    )
    plt.title(f"Italy: Top Critical Import Routes by Vulnerability Score, {YEAR}")
    plt.xlabel("Route vulnerability score")
    plt.ylabel("Supplier | Product")
    save_fig(os.path.join(FIG_DIR, f"italy_top_critical_import_routes_{YEAR}.png"))


# ------------------------------------------------------------
# 15.14 Save figures: Italy supplier exposure
# ------------------------------------------------------------

if "critical_routes_by_supplier" in globals() and isinstance(critical_routes_by_supplier, pd.DataFrame) and not critical_routes_by_supplier.empty:
    supplier_plot = critical_routes_by_supplier.head(20).copy()

    plt.figure(figsize=(10, 7))
    plt.barh(
        supplier_plot["supplier_country"][::-1],
        supplier_plot["total_route_score"][::-1]
    )
    plt.title(f"Italy: Critical Import Exposure by Supplier Country, {YEAR}")
    plt.xlabel("Total route vulnerability score")
    plt.ylabel("Supplier country")
    save_fig(os.path.join(FIG_DIR, f"italy_supplier_country_exposure_{YEAR}.png"))


# ------------------------------------------------------------
# 15.15 Save metadata
# ------------------------------------------------------------

metadata_path = os.path.join(OUT_DIR, f"metadata_slack_analysis_{YEAR}.txt")

with open(metadata_path, "w") as f:
    f.write("BACI Country-Product Slack Analysis\n")
    f.write("===================================\n\n")
    f.write(f"Created at: {datetime.now()}\n")
    f.write(f"Year: {YEAR}\n")
    f.write(f"Theta: {THETA}\n")

    if "MIN_WEIGHT" in globals():
        f.write(f"Minimum edge weight: {MIN_WEIGHT}\n")

    f.write("\nGlobal exporter-product definitions:\n")
    f.write("w_cp = exports of country c in product p, aggregated over importers.\n")
    f.write("T_p = total observed exports of product p in the thresholded network.\n")
    f.write("Q_p = theta * T_p.\n")
    f.write("Slack_cp = (T_p - w_cp) / Q_p.\n")
    f.write("Critical edge if Slack_cp < 1.\n")
    f.write("With theta = 0.70, this is equivalent to supply share > 30%.\n")
    f.write("Country weighted slack = sum_p beta_cp * Slack_cp.\n")
    f.write("beta_cp = w_cp / total exports of country c.\n")

    f.write("\nProduct-level fragility definitions:\n")
    f.write("Product-level weighted slack is not used as main metric because it is mechanically related to HHI.\n")
    f.write("criticality_depth_cp = max(0, 1 - Slack_cp).\n")
    f.write("product_fragility_score_p = sum_c supply_share_cp * criticality_depth_cp.\n")
    f.write("HHI is retained as a benchmark concentration measure.\n")

    f.write("\nItaly import-side definitions:\n")
    f.write("For Italy as importer, w_cp = imports of product p from supplier c.\n")
    f.write("T_import_p = Italy's total imports of product p.\n")
    f.write("Slack_import_cp = (T_import_p - w_cp) / (theta * T_import_p).\n")
    f.write("Critical import route if Slack_import_cp < 1.\n")
    f.write("route_vulnerability_score = supplier_share * import_criticality_depth * log(1 + weight).\n")

    f.write("\nInterpretation:\n")
    f.write("Slack measures structural dependence under a quota rule, not geopolitical risk directly.\n")
    f.write("Geopolitical vulnerability emerges when structural dependence overlaps with strategic sectors, political risk, logistical disruption, or low substitutability.\n")

print("Saved metadata:", metadata_path)


# ------------------------------------------------------------
# 15.16 Save Python objects as pickle
# ------------------------------------------------------------

objects_to_save = {}

objects_list = [
    "edges_cp",
    "country_slack",
    "product_slack",
    "critical_edges",
    "criticality_contribution_edges",
    "top_pivotal_countries_by_slack",
    "top_fragile_products",
    "systemic_fragile_products",
    "hs2_fragility_summary",
    "italy_critical_products",
    "italy_import_vulnerability",
    "italy_critical_routes",
    "italy_critical_route_table",
    "critical_routes_by_supplier",
    "supplier_map_plot",
    "italy_product_specific_map_data",
    "italy_bipartite_subgraph",
    "supplier_sector",
    "sector_italy"
]

for name in objects_list:
    if name in globals():
        objects_to_save[name] = globals()[name]

pickle_path = os.path.join(
    OUT_DIR,
    f"slack_analysis_objects_{YEAR}.pkl"
)

with open(pickle_path, "wb") as f:
    pickle.dump(objects_to_save, f)

print("Saved pickle object:", pickle_path)


# ------------------------------------------------------------
# 15.17 Final report
# ------------------------------------------------------------

print("\n============================================================")
print("SAVE COMPLETE")
print("============================================================")
print("Main output folder:", OUT_DIR)
print("Tables folder:", TABLE_DIR)
print("Figures folder:", FIG_DIR)
print("Excel workbook:", excel_path)
print("Metadata:", metadata_path)
print("Pickle:", pickle_path)
print("============================================================")

# DATA SAVED

In [ ]:
# ============================================================
# 16. ROBUSTNESS AND SENSITIVITY ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from datetime import datetime

# ------------------------------------------------------------
# 16.0 Settings
# ------------------------------------------------------------

THETA_VALUES = [0.60, 0.65, 0.70, 0.75, 0.80]
MIN_WEIGHT_VALUES = [0, 50, 100, 500, 1000]

BASELINE_THETA = THETA if "THETA" in globals() else 0.70
BASELINE_MIN_WEIGHT = MIN_WEIGHT if "MIN_WEIGHT" in globals() else 100

ROBUST_DIR = os.path.join(OUT_DIR, "robustness")
ROBUST_TABLE_DIR = os.path.join(ROBUST_DIR, "tables")
ROBUST_FIG_DIR = os.path.join(ROBUST_DIR, "figures")

os.makedirs(ROBUST_DIR, exist_ok=True)
os.makedirs(ROBUST_TABLE_DIR, exist_ok=True)
os.makedirs(ROBUST_FIG_DIR, exist_ok=True)

print("Robustness output folder:", ROBUST_DIR)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def risk_label_from_slack(slack_series):
    return np.select(
        [
            slack_series < 0.5,
            slack_series < 1.0,
            slack_series < 1.1,
            slack_series < 1.2
        ],
        [
            "SEVERE",
            "CRITICAL",
            "NEAR-CRITICAL",
            "WATCH"
        ],
        default="REDUNDANT"
    )


def save_table(df, filename):
    path = os.path.join(ROBUST_TABLE_DIR, filename)
    df.to_csv(path, index=False)
    print("Saved table:", path)


def save_plot(filename):
    path = os.path.join(ROBUST_FIG_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)


def jaccard_similarity(a, b):
    a = set(a)
    b = set(b)
    if len(a) == 0 and len(b) == 0:
        return 1.0
    if len(a.union(b)) == 0:
        return np.nan
    return len(a.intersection(b)) / len(a.union(b))


# ============================================================
# A. GLOBAL THETA ROBUSTNESS FUNCTIONS
# ============================================================

def compute_global_edges_for_theta(country_product_df, theta=0.70, min_weight=100):
    df = country_product_df.copy()
    df = df[df["weight"] >= min_weight].copy()

    df["year"] = YEAR
    df["theta"] = theta
    df["min_weight"] = min_weight

    df["T_p"] = df.groupby("hs4")["weight"].transform("sum")
    df["Q_p"] = theta * df["T_p"]
    df["supply_share_cp"] = df["weight"] / df["T_p"]

    df["Slack_cp"] = (df["T_p"] - df["weight"]) / df["Q_p"]
    df["slack_gap"] = df["Slack_cp"] - 1
    df["is_critical_supplier"] = df["Slack_cp"] < 1
    df["criticality_depth"] = np.maximum(0, 1 - df["Slack_cp"])
    df["risk_label"] = risk_label_from_slack(df["Slack_cp"])

    df["s_c"] = df.groupby(["i", "country_name"])["weight"].transform("sum")
    df["beta_cp"] = df["weight"] / df["s_c"]
    df["country_weighted_slack_contribution"] = df["beta_cp"] * df["Slack_cp"]
    df["country_criticality_contribution"] = df["beta_cp"] * df["criticality_depth"]

    return df


def summarize_countries_for_theta(edges_df):
    country_summary = (
        edges_df
        .groupby(["year", "theta", "min_weight", "i", "country_name"], as_index=False)
        .agg(
            total_exports=("weight", "sum"),
            n_products=("hs4", "nunique"),
            n_critical_products=("is_critical_supplier", "sum"),
            n_severe_products=("Slack_cp", lambda x: (x < 0.5).sum()),
            Slack_mean=("Slack_cp", "mean"),
            Slack_min=("Slack_cp", "min"),
            Slack_weighted=("country_weighted_slack_contribution", "sum"),
            weighted_criticality_depth=("country_criticality_contribution", "sum"),
            max_supply_share=("supply_share_cp", "max")
        )
    )

    extra = (
        edges_df
        .assign(
            critical_export_value=lambda d: np.where(d["Slack_cp"] < 1, d["weight"], 0),
            severe_export_value=lambda d: np.where(d["Slack_cp"] < 0.5, d["weight"], 0)
        )
        .groupby(["year", "theta", "min_weight", "i", "country_name"], as_index=False)
        .agg(
            critical_export_value=("critical_export_value", "sum"),
            severe_export_value=("severe_export_value", "sum")
        )
    )

    country_summary = country_summary.merge(
        extra,
        on=["year", "theta", "min_weight", "i", "country_name"],
        how="left"
    )

    country_summary["critical_product_share"] = (
        country_summary["n_critical_products"] / country_summary["n_products"]
    )

    country_summary["critical_export_share"] = (
        country_summary["critical_export_value"] / country_summary["total_exports"]
    )

    country_summary["rank_weighted_slack"] = country_summary["Slack_weighted"].rank(
        method="min", ascending=True
    )
    country_summary["rank_critical_products"] = country_summary["n_critical_products"].rank(
        method="min", ascending=False
    )
    country_summary["rank_critical_export_value"] = country_summary["critical_export_value"].rank(
        method="min", ascending=False
    )

    country_summary["combined_country_rank"] = (
        country_summary["rank_weighted_slack"]
        + country_summary["rank_critical_products"]
        + country_summary["rank_critical_export_value"]
    )

    return country_summary.sort_values(
        ["combined_country_rank", "Slack_weighted", "n_critical_products"],
        ascending=[True, True, False]
    )


def summarize_products_for_theta(edges_df):
    edges_df = edges_df.copy()

    edges_df["critical_supplier_share_contribution"] = (
        edges_df["supply_share_cp"] *
        edges_df["is_critical_supplier"].astype(int)
    )

    edges_df["product_fragility_contribution"] = (
        edges_df["supply_share_cp"] *
        edges_df["criticality_depth"]
    )

    product_summary = (
        edges_df
        .groupby(["year", "theta", "min_weight", "hs4", "hs2", "product_description"], as_index=False)
        .agg(
            total_trade=("weight", "sum"),
            n_exporters=("country_name", "nunique"),
            max_supply_share=("supply_share_cp", "max"),
            HHI=("supply_share_cp", lambda x: np.sum(np.square(x))),
            entropy=("supply_share_cp", lambda x: -np.sum(x * np.log(x + 1e-12))),
            Slack_mean=("Slack_cp", "mean"),
            Slack_min=("Slack_cp", "min"),
            n_critical_suppliers=("is_critical_supplier", "sum"),
            n_severe_suppliers=("Slack_cp", lambda x: (x < 0.5).sum()),
            critical_supplier_share=("critical_supplier_share_contribution", "sum"),
            product_fragility_score=("product_fragility_contribution", "sum"),
            max_criticality_depth=("criticality_depth", "max")
        )
    )

    product_summary["effective_n_exporters"] = 1 / product_summary["HHI"]
    product_summary["has_critical_supplier"] = product_summary["n_critical_suppliers"] > 0

    dominant_idx = edges_df.groupby(["year", "theta", "min_weight", "hs4"])["supply_share_cp"].idxmax()

    dominant_supplier = (
        edges_df
        .loc[
            dominant_idx,
            ["year", "theta", "min_weight", "hs4", "country_name", "supply_share_cp", "Slack_cp", "risk_label"]
        ]
        .rename(columns={
            "country_name": "dominant_supplier",
            "supply_share_cp": "dominant_supplier_share",
            "Slack_cp": "dominant_supplier_slack",
            "risk_label": "dominant_supplier_risk"
        })
    )

    product_summary = product_summary.merge(
        dominant_supplier,
        on=["year", "theta", "min_weight", "hs4"],
        how="left"
    )

    return product_summary


def compute_systemic_products_for_theta(product_summary):
    df = product_summary.query("n_critical_suppliers > 0").copy()

    if df.empty:
        return df

    df["rank_fragility"] = df["product_fragility_score"].rank(
        method="min", ascending=False
    )
    df["rank_trade"] = df["total_trade"].rank(
        method="min", ascending=False
    )
    df["systemic_fragility_rank"] = df["rank_fragility"] + df["rank_trade"]

    return df.sort_values(
        ["systemic_fragility_rank", "product_fragility_score", "total_trade"],
        ascending=[True, False, False]
    )


# ============================================================
# B. ITALY IMPORT THETA ROBUSTNESS FUNCTIONS
# ============================================================

def get_country_code_robust(country_lookup_df, query):
    exact = country_lookup_df[
        country_lookup_df["country_name"].str.lower() == query.lower()
    ].copy()

    if not exact.empty:
        return int(exact.iloc[0]["country_code"])

    matches = country_lookup_df[
        country_lookup_df["country_name"].str.contains(query, case=False, na=False)
    ].copy()

    if matches.empty:
        raise ValueError(f"No country code found for query: {query}")

    display(matches.head(20))
    return int(matches.iloc[0]["country_code"])


def build_import_base_from_baci(
    baci_file,
    country_lookup_df,
    importer_query="Italy",
    chunksize=1_000_000
):
    """
    Builds an unfiltered supplier-HS4 import table for the selected importer.
    This is done once and then reused for theta and min-weight robustness.
    """

    importer_code = get_country_code_robust(country_lookup_df, importer_query)
    chunks = []

    for chunk in pd.read_csv(
        baci_file,
        usecols=["t", "i", "j", "k", "v"],
        chunksize=chunksize
    ):
        chunk = chunk[chunk["j"] == importer_code].copy()

        if chunk.empty:
            continue

        chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce")
        chunk = chunk.dropna(subset=["i", "k", "v"])
        chunk = chunk[chunk["v"] > 0].copy()

        if chunk.empty:
            continue

        chunk["i"] = chunk["i"].astype(int)
        chunk["k"] = (
            chunk["k"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(6)
        )
        chunk["hs4"] = chunk["k"].str[:4]
        chunk["hs2"] = chunk["k"].str[:2]

        agg = (
            chunk
            .groupby(["i", "hs4", "hs2"], as_index=False)
            .agg(
                weight=("v", "sum"),
                n_records=("v", "size")
            )
        )
        chunks.append(agg)

    if len(chunks) == 0:
        raise ValueError("No import records found.")

    imports_base = pd.concat(chunks, ignore_index=True)

    imports_base = (
        imports_base
        .groupby(["i", "hs4", "hs2"], as_index=False)
        .agg(
            weight=("weight", "sum"),
            n_records=("n_records", "sum")
        )
    )

    imports_base = imports_base.merge(
        country_lookup_df,
        left_on="i",
        right_on="country_code",
        how="left"
    )

    imports_base = imports_base.rename(columns={"country_name": "supplier_country"})
    imports_base["supplier_country"] = imports_base["supplier_country"].fillna(
        "C_" + imports_base["i"].astype(str)
    )

    # Add product descriptions if available from existing global table
    if "edges_cp" in globals() and "product_description" in edges_cp.columns:
        product_desc_lookup = (
            edges_cp[["hs4", "product_description"]]
            .drop_duplicates("hs4")
        )
        imports_base = imports_base.merge(product_desc_lookup, on="hs4", how="left")
    elif "country_product" in globals() and "product_description" in country_product.columns:
        product_desc_lookup = (
            country_product[["hs4", "product_description"]]
            .drop_duplicates("hs4")
        )
        imports_base = imports_base.merge(product_desc_lookup, on="hs4", how="left")
    else:
        imports_base["product_description"] = imports_base["hs4"]

    imports_base["product_description"] = imports_base["product_description"].fillna(
        imports_base["hs4"]
    )

    imports_base["year"] = YEAR
    imports_base["importer_country"] = importer_query

    if "country_code" in imports_base.columns:
        imports_base = imports_base.drop(columns=["country_code"])

    return imports_base


def compute_import_vulnerability_from_base(imports_base, theta=0.70, min_weight=100):
    imports = imports_base.copy()
    imports = imports[imports["weight"] >= min_weight].copy()

    imports["theta"] = theta
    imports["min_weight"] = min_weight

    imports["T_import_p"] = imports.groupby("hs4")["weight"].transform("sum")
    imports["Q_import_p"] = theta * imports["T_import_p"]

    imports["supplier_share"] = imports["weight"] / imports["T_import_p"]

    imports["Slack_import_cp"] = (
        imports["T_import_p"] - imports["weight"]
    ) / imports["Q_import_p"]

    imports["import_slack_gap"] = imports["Slack_import_cp"] - 1
    imports["is_critical_import_supplier"] = imports["Slack_import_cp"] < 1
    imports["import_criticality_depth"] = np.maximum(0, 1 - imports["Slack_import_cp"])
    imports["risk_label"] = risk_label_from_slack(imports["Slack_import_cp"])

    imports["route_vulnerability_score"] = (
        imports["supplier_share"]
        * imports["import_criticality_depth"]
        * np.log1p(imports["weight"])
    )

    imports["trade_value_musd"] = imports["weight"] / 1000

    return imports.sort_values(
        ["route_vulnerability_score", "supplier_share", "weight"],
        ascending=[False, False, False]
    )


def summarize_italy_import_for_theta(imports_df):
    crit = imports_df[imports_df["Slack_import_cp"] < 1].copy()

    if crit.empty:
        top_suppliers = ""
        top_routes = ""
    else:
        supplier_scores = (
            crit.groupby("supplier_country", as_index=False)
            .agg(total_route_score=("route_vulnerability_score", "sum"))
            .sort_values("total_route_score", ascending=False)
            .head(5)
        )
        top_suppliers = " | ".join(
            supplier_scores["supplier_country"].astype(str).tolist()
        )

        top_routes = " | ".join(
            (
                crit.head(5)["supplier_country"].astype(str)
                + " - "
                + crit.head(5)["hs4"].astype(str)
            ).tolist()
        )

    out = {
        "theta": imports_df["theta"].iloc[0],
        "min_weight": imports_df["min_weight"].iloc[0],
        "n_import_links": imports_df.shape[0],
        "n_supplier_countries": imports_df["supplier_country"].nunique(),
        "n_imported_hs4": imports_df["hs4"].nunique(),
        "n_critical_import_routes": (imports_df["Slack_import_cp"] < 1).sum(),
        "share_critical_import_routes": (imports_df["Slack_import_cp"] < 1).mean(),
        "n_severe_import_routes": (imports_df["Slack_import_cp"] < 0.5).sum(),
        "n_critical_supplier_countries": crit["supplier_country"].nunique() if not crit.empty else 0,
        "n_critical_import_products": crit["hs4"].nunique() if not crit.empty else 0,
        "total_critical_imports_musd": crit["trade_value_musd"].sum() if not crit.empty else 0,
        "total_route_vulnerability_score": crit["route_vulnerability_score"].sum() if not crit.empty else 0,
        "top_suppliers_by_score": top_suppliers,
        "top_routes_by_score": top_routes
    }

    return out


# ============================================================
# A. THETA ROBUSTNESS FOR ITALY IMPORT VULNERABILITY
# ============================================================

print("\n============================================================")
print("A. Theta robustness for Italy import vulnerability")
print("============================================================")

# Build unfiltered Italy import base once.
# This may take some time, but avoids rereading BACI repeatedly for every theta.
italy_import_base = build_import_base_from_baci(
    baci_file=baci_file,
    country_lookup_df=country_lookup,
    importer_query="Italy",
    chunksize=1_000_000
)

italy_theta_tables = {}
italy_theta_summary_rows = []

for theta in THETA_VALUES:
    tmp = compute_import_vulnerability_from_base(
        italy_import_base,
        theta=theta,
        min_weight=BASELINE_MIN_WEIGHT
    )

    italy_theta_tables[theta] = tmp
    italy_theta_summary_rows.append(summarize_italy_import_for_theta(tmp))

italy_theta_summary = pd.DataFrame(italy_theta_summary_rows)

display(italy_theta_summary)
save_table(italy_theta_summary, f"italy_import_theta_robustness_{YEAR}.csv")

plt.figure(figsize=(8, 5))
plt.plot(
    italy_theta_summary["theta"],
    italy_theta_summary["n_critical_import_routes"],
    marker="o"
)
plt.title(f"Italy import vulnerability by theta, {YEAR}")
plt.xlabel("Theta")
plt.ylabel("Number of critical import routes")
save_plot(f"italy_import_critical_routes_by_theta_{YEAR}.png")


# ============================================================
# B. THETA ROBUSTNESS FOR COUNTRY RANKINGS
# ============================================================

print("\n============================================================")
print("B. Theta robustness for global country rankings")
print("============================================================")

global_edge_tables = {}
country_theta_tables = {}
product_theta_tables = {}
systemic_product_theta_tables = {}

global_theta_summary_rows = []
top_countries_by_theta = []

for theta in THETA_VALUES:
    edges_theta = compute_global_edges_for_theta(
        country_product,
        theta=theta,
        min_weight=BASELINE_MIN_WEIGHT
    )

    country_theta = summarize_countries_for_theta(edges_theta)
    product_theta = summarize_products_for_theta(edges_theta)
    systemic_theta = compute_systemic_products_for_theta(product_theta)

    global_edge_tables[theta] = edges_theta
    country_theta_tables[theta] = country_theta
    product_theta_tables[theta] = product_theta
    systemic_product_theta_tables[theta] = systemic_theta

    global_theta_summary_rows.append({
        "theta": theta,
        "min_weight": BASELINE_MIN_WEIGHT,
        "n_edges": edges_theta.shape[0],
        "n_countries": edges_theta["country_name"].nunique(),
        "n_products": edges_theta["hs4"].nunique(),
        "n_critical_edges": edges_theta["is_critical_supplier"].sum(),
        "share_critical_edges": edges_theta["is_critical_supplier"].mean(),
        "n_critical_countries": edges_theta.loc[edges_theta["is_critical_supplier"], "country_name"].nunique(),
        "n_critical_products": edges_theta.loc[edges_theta["is_critical_supplier"], "hs4"].nunique(),
        "n_severe_edges": (edges_theta["Slack_cp"] < 0.5).sum()
    })

    top_tmp = country_theta.head(10).copy()
    top_tmp["top_rank_position"] = range(1, len(top_tmp) + 1)
    top_countries_by_theta.append(top_tmp)

global_theta_summary = pd.DataFrame(global_theta_summary_rows)
top_countries_by_theta = pd.concat(top_countries_by_theta, ignore_index=True)

display(global_theta_summary)
display(
    top_countries_by_theta[
        [
            "theta",
            "top_rank_position",
            "country_name",
            "combined_country_rank",
            "Slack_weighted",
            "n_critical_products",
            "critical_export_value",
            "critical_export_share"
        ]
    ]
)

save_table(global_theta_summary, f"global_theta_summary_{YEAR}.csv")
save_table(top_countries_by_theta, f"top_countries_by_theta_{YEAR}.csv")

plt.figure(figsize=(8, 5))
plt.plot(
    global_theta_summary["theta"],
    global_theta_summary["n_critical_edges"],
    marker="o"
)
plt.title(f"Global critical country-product edges by theta, {YEAR}")
plt.xlabel("Theta")
plt.ylabel("Number of critical edges")
save_plot(f"global_critical_edges_by_theta_{YEAR}.png")

# Country rank stability table
selected_countries = sorted(
    top_countries_by_theta["country_name"].dropna().unique().tolist()
)

country_rank_stability = []

for theta, df in country_theta_tables.items():
    tmp = df[df["country_name"].isin(selected_countries)][
        [
            "theta",
            "country_name",
            "combined_country_rank",
            "Slack_weighted",
            "n_critical_products",
            "critical_export_value"
        ]
    ].copy()
    country_rank_stability.append(tmp)

country_rank_stability = pd.concat(country_rank_stability, ignore_index=True)

country_rank_pivot = country_rank_stability.pivot_table(
    index="country_name",
    columns="theta",
    values="combined_country_rank",
    aggfunc="min"
)

display(country_rank_pivot)
save_table(country_rank_stability, f"country_rank_stability_by_theta_{YEAR}.csv")


# ============================================================
# C. THETA ROBUSTNESS FOR SYSTEMIC FRAGILE PRODUCTS
# ============================================================

print("\n============================================================")
print("C. Theta robustness for systemic fragile products")
print("============================================================")

top_systemic_products_by_theta = []

for theta, df in systemic_product_theta_tables.items():
    if df.empty:
        continue

    top_tmp = df.head(10).copy()
    top_tmp["top_rank_position"] = range(1, len(top_tmp) + 1)
    top_systemic_products_by_theta.append(top_tmp)

top_systemic_products_by_theta = pd.concat(top_systemic_products_by_theta, ignore_index=True)

display(
    top_systemic_products_by_theta[
        [
            "theta",
            "top_rank_position",
            "hs4",
            "hs2",
            "product_description",
            "total_trade",
            "dominant_supplier",
            "dominant_supplier_share",
            "product_fragility_score",
            "rank_fragility",
            "rank_trade",
            "systemic_fragility_rank"
        ]
    ]
)

save_table(top_systemic_products_by_theta, f"top_systemic_products_by_theta_{YEAR}.csv")

# Product rank stability for products appearing in top 10 for at least one theta
selected_products = sorted(
    top_systemic_products_by_theta["hs4"].dropna().unique().tolist()
)

product_rank_stability = []

for theta, df in systemic_product_theta_tables.items():
    tmp = df[df["hs4"].isin(selected_products)][
        [
            "theta",
            "hs4",
            "product_description",
            "dominant_supplier",
            "product_fragility_score",
            "rank_fragility",
            "rank_trade",
            "systemic_fragility_rank"
        ]
    ].copy()
    product_rank_stability.append(tmp)

product_rank_stability = pd.concat(product_rank_stability, ignore_index=True)

product_rank_pivot = product_rank_stability.pivot_table(
    index="hs4",
    columns="theta",
    values="systemic_fragility_rank",
    aggfunc="min"
)

display(product_rank_pivot)
save_table(product_rank_stability, f"product_rank_stability_by_theta_{YEAR}.csv")


# ============================================================
# D. MINIMUM TRADE THRESHOLD ROBUSTNESS
# ============================================================

print("\n============================================================")
print("D. Minimum trade threshold robustness")
print("============================================================")

min_weight_summary_rows = []

for mw in MIN_WEIGHT_VALUES:

    # Global network at baseline theta
    edges_mw = compute_global_edges_for_theta(
        country_product,
        theta=BASELINE_THETA,
        min_weight=mw
    )

    # Italy imports at baseline theta
    italy_mw = compute_import_vulnerability_from_base(
        italy_import_base,
        theta=BASELINE_THETA,
        min_weight=mw
    )

    italy_mw_crit = italy_mw[italy_mw["Slack_import_cp"] < 1].copy()

    min_weight_summary_rows.append({
        "min_weight": mw,
        "theta": BASELINE_THETA,

        # Global exporter-product network
        "global_n_edges": edges_mw.shape[0],
        "global_n_countries": edges_mw["country_name"].nunique(),
        "global_n_products": edges_mw["hs4"].nunique(),
        "global_n_critical_edges": edges_mw["is_critical_supplier"].sum(),
        "global_share_critical_edges": edges_mw["is_critical_supplier"].mean(),
        "global_n_critical_countries": edges_mw.loc[edges_mw["is_critical_supplier"], "country_name"].nunique(),
        "global_n_critical_products": edges_mw.loc[edges_mw["is_critical_supplier"], "hs4"].nunique(),

        # Italy import network
        "italy_n_import_links": italy_mw.shape[0],
        "italy_n_supplier_countries": italy_mw["supplier_country"].nunique(),
        "italy_n_imported_hs4": italy_mw["hs4"].nunique(),
        "italy_n_critical_routes": italy_mw_crit.shape[0],
        "italy_share_critical_routes": (italy_mw["Slack_import_cp"] < 1).mean(),
        "italy_n_critical_suppliers": italy_mw_crit["supplier_country"].nunique(),
        "italy_n_critical_products": italy_mw_crit["hs4"].nunique(),
        "italy_total_critical_imports_musd": italy_mw_crit["trade_value_musd"].sum()
    })

min_weight_summary = pd.DataFrame(min_weight_summary_rows)

display(min_weight_summary)
save_table(min_weight_summary, f"min_weight_robustness_{YEAR}.csv")

plt.figure(figsize=(8, 5))
plt.plot(
    min_weight_summary["min_weight"],
    min_weight_summary["global_n_critical_edges"],
    marker="o",
    label="Global critical edges"
)
plt.plot(
    min_weight_summary["min_weight"],
    min_weight_summary["italy_n_critical_routes"],
    marker="o",
    label="Italy critical import routes"
)
plt.title(f"Critical links by minimum trade threshold, theta={BASELINE_THETA}, {YEAR}")
plt.xlabel("Minimum trade threshold")
plt.ylabel("Number of critical links")
plt.legend()
save_plot(f"critical_links_by_min_weight_{YEAR}.png")


# ============================================================
# E. JACCARD OVERLAP STABILITY
# ============================================================

print("\n============================================================")
print("E. Jaccard overlap stability across theta values")
print("============================================================")

set_results = {}

for theta in THETA_VALUES:
    edges_theta = global_edge_tables[theta]
    italy_theta = italy_theta_tables[theta]

    global_critical = edges_theta[edges_theta["Slack_cp"] < 1].copy()
    italy_critical = italy_theta[italy_theta["Slack_import_cp"] < 1].copy()

    set_results[theta] = {
        "global_edges": set(zip(global_critical["country_name"], global_critical["hs4"])),
        "global_countries": set(global_critical["country_name"]),
        "global_products": set(global_critical["hs4"]),

        "italy_import_routes": set(zip(italy_critical["supplier_country"], italy_critical["hs4"])),
        "italy_supplier_countries": set(italy_critical["supplier_country"]),
        "italy_import_products": set(italy_critical["hs4"])
    }

jaccard_rows = []

set_types = [
    "global_edges",
    "global_countries",
    "global_products",
    "italy_import_routes",
    "italy_supplier_countries",
    "italy_import_products"
]

for set_type in set_types:
    for theta_a in THETA_VALUES:
        for theta_b in THETA_VALUES:
            jaccard_rows.append({
                "set_type": set_type,
                "theta_a": theta_a,
                "theta_b": theta_b,
                "jaccard": jaccard_similarity(
                    set_results[theta_a][set_type],
                    set_results[theta_b][set_type]
                )
            })

jaccard_results = pd.DataFrame(jaccard_rows)

display(jaccard_results.head(20))
save_table(jaccard_results, f"jaccard_overlap_by_theta_{YEAR}.csv")

# Plot Jaccard heatmaps using matplotlib
for set_type in set_types:
    pivot = jaccard_results.query("set_type == @set_type").pivot(
        index="theta_a",
        columns="theta_b",
        values="jaccard"
    )

    plt.figure(figsize=(6, 5))
    plt.imshow(pivot.values, aspect="auto")
    plt.colorbar(label="Jaccard similarity")
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title(f"Jaccard stability: {set_type}")
    plt.xlabel("Theta")
    plt.ylabel("Theta")

    # Add values inside cells
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            plt.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center")

    save_plot(f"jaccard_{set_type}_{YEAR}.png")


# ============================================================
# F. HHI BENCHMARK COMPARISON
# ============================================================

print("\n============================================================")
print("F. HHI benchmark comparison")
print("============================================================")

baseline_product_summary = product_theta_tables.get(
    BASELINE_THETA,
    product_slack if "product_slack" in globals() else None
)

if baseline_product_summary is None:
    raise ValueError("No baseline product summary found.")

hhi_cols = [
    "HHI",
    "max_supply_share",
    "critical_supplier_share",
    "product_fragility_score",
    "Slack_min",
    "n_critical_suppliers",
    "total_trade"
]

hhi_cols = [c for c in hhi_cols if c in baseline_product_summary.columns]

hhi_correlation = baseline_product_summary[hhi_cols].corr()

print("Correlation matrix:")
display(hhi_correlation)

hhi_correlation.to_csv(
    os.path.join(ROBUST_TABLE_DIR, f"hhi_correlation_matrix_{YEAR}.csv")
)

plt.figure(figsize=(7, 5))
plt.scatter(
    baseline_product_summary["HHI"],
    baseline_product_summary["product_fragility_score"],
    alpha=0.5
)
plt.title(f"HHI vs threshold-sensitive product fragility, {YEAR}")
plt.xlabel("HHI")
plt.ylabel("Product fragility score")
save_plot(f"hhi_vs_product_fragility_robustness_{YEAR}.png")

plt.figure(figsize=(7, 5))
plt.scatter(
    baseline_product_summary["max_supply_share"],
    baseline_product_summary["product_fragility_score"],
    alpha=0.5
)
plt.title(f"Dominant supplier share vs product fragility, {YEAR}")
plt.xlabel("Dominant supplier share")
plt.ylabel("Product fragility score")
save_plot(f"dominant_supplier_share_vs_product_fragility_{YEAR}.png")

# HHI bin summary
hhi_bin_summary = (
    baseline_product_summary
    .assign(
        HHI_bin=pd.qcut(
            baseline_product_summary["HHI"],
            q=5,
            duplicates="drop"
        )
    )
    .groupby("HHI_bin", observed=True)
    .agg(
        n_products=("hs4", "nunique"),
        avg_HHI=("HHI", "mean"),
        avg_product_fragility=("product_fragility_score", "mean"),
        median_product_fragility=("product_fragility_score", "median"),
        share_products_with_critical_suppliers=("has_critical_supplier", "mean"),
        avg_critical_supplier_share=("critical_supplier_share", "mean"),
        avg_max_supply_share=("max_supply_share", "mean")
    )
    .reset_index()
)

display(hhi_bin_summary)
save_table(hhi_bin_summary, f"hhi_bin_summary_{YEAR}.csv")


# ============================================================
# FINAL ROBUSTNESS REPORT
# ============================================================

robust_metadata_path = os.path.join(ROBUST_DIR, f"robustness_metadata_{YEAR}.txt")

with open(robust_metadata_path, "w") as f:
    f.write("Robustness analysis for BACI slack framework\n")
    f.write("===========================================\n\n")
    f.write(f"Created at: {datetime.now()}\n")
    f.write(f"Year: {YEAR}\n")
    f.write(f"Baseline theta: {BASELINE_THETA}\n")
    f.write(f"Theta values: {THETA_VALUES}\n")
    f.write(f"Baseline minimum trade threshold: {BASELINE_MIN_WEIGHT}\n")
    f.write(f"Minimum trade thresholds: {MIN_WEIGHT_VALUES}\n\n")
    f.write("Sections:\n")
    f.write("A. Theta robustness for Italy import vulnerability.\n")
    f.write("B. Theta robustness for global country rankings.\n")
    f.write("C. Theta robustness for systemic fragile products.\n")
    f.write("D. Minimum trade threshold robustness.\n")
    f.write("E. Jaccard overlap stability.\n")
    f.write("F. HHI benchmark comparison.\n")

print("\n============================================================")
print("ROBUSTNESS ANALYSIS COMPLETE")
print("============================================================")
print("Robustness folder:", ROBUST_DIR)
print("Tables:", ROBUST_TABLE_DIR)
print("Figures:", ROBUST_FIG_DIR)
print("Metadata:", robust_metadata_path)
print("============================================================")

In [ ]:
jaccard_results.groupby("set_type")["jaccard"].describe()


In [ ]:
jaccard_results.query("theta_a == 0.70")[["set_type", "theta_b", "jaccard"]]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr
import os

THETA_VALUES      = [0.60, 0.65, 0.70, 0.75, 0.80]
BASELINE_THETA    = 0.70
BASELINE_MIN_WEIGHT = 100

# ── re-use the ROBUST_FIG_DIR / ROBUST_TABLE_DIR set in cell 16 ──────────────
os.makedirs(ROBUST_FIG_DIR,   exist_ok=True)
os.makedirs(ROBUST_TABLE_DIR, exist_ok=True)

# Concatenate all country and product summaries from cell 16 for ease of use
all_theta_country_summary = pd.concat(country_theta_tables.values(), ignore_index=True)
all_theta_product_summary = pd.concat(product_theta_tables.values(), ignore_index=True)

# In case italy robustness was not run, handle the potential NameError for all_theta_italy_routes
if 'italy_theta_tables' in globals():
    all_theta_italy_routes = pd.concat(italy_theta_tables.values(), ignore_index=True)
else:
    all_theta_italy_routes = pd.DataFrame() # Create an empty DataFrame if not available

# ============================================================
# A. JACCARD HEATMAPS
# ============================================================

set_types = jaccard_results["set_type"].unique()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

cmap = sns.color_palette("YlOrRd_r", as_cmap=True)   # low Jaccard = warm, high = cool

for ax, stype in zip(axes, set_types):

    sub = jaccard_results[jaccard_results["set_type"] == stype].copy()

    # Build symmetric 5x5 matrix
    mat = pd.DataFrame(
        index=THETA_VALUES,
        columns=THETA_VALUES,
        dtype=float
    )

    for _, row in sub.iterrows():
        mat.loc[row["theta_a"], row["theta_b"]] = row["jaccard"]
        mat.loc[row["theta_b"], row["theta_a"]] = row["jaccard"]   # symmetric

    mat = mat.astype(float)

    sns.heatmap(
        mat,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",     # green = stable, red = unstable
        vmin=0.0,
        vmax=1.0,
        linewidths=0.5,
        linecolor="white",
        cbar=False,
        annot_kws={"size": 9}
    )

    # Highlight baseline row/col
    baseline_idx = THETA_VALUES.index(BASELINE_THETA)
    ax.axhline(y=baseline_idx,     color="steelblue", lw=2.5, ls="--")
    ax.axhline(y=baseline_idx + 1, color="steelblue", lw=2.5, ls="--")
    ax.axvline(x=baseline_idx,     color="steelblue", lw=2.5, ls="--")
    ax.axvline(x=baseline_idx + 1, color="steelblue", lw=2.5, ls="--")

    ax.set_title(stype.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.set_xlabel("θ", fontsize=10)
    ax.set_ylabel("θ", fontsize=10)
    ax.set_xticklabels([str(t) for t in THETA_VALUES], fontsize=9)
    ax.set_yticklabels([str(t) for t in THETA_VALUES], fontsize=9, rotation=0)

# Hide the unused 6th subplot if set_types has only 5
if len(set_types) < 6:
    axes[5].set_visible(False)

fig.suptitle(
    f"Jaccard Similarity of Critical Sets Across θ Values  |  Year {YEAR}\n"
    f"(Blue dashed lines = baseline θ = {BASELINE_THETA})",
    fontsize=13,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
heatmap_path = os.path.join(ROBUST_FIG_DIR, f"jaccard_heatmaps_all_sets_{YEAR}.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", heatmap_path)


# ============================================================
# B. SPEARMAN RANK CORRELATION ACROSS THETA VALUES
#
# For each theta pair, compute Spearman rho on:
#   - Slack_weighted   (country level)
#   - product_fragility_score  (product level)
#
# This answers: does the RANKING change even when the same
# countries/products remain in the critical set?
# ============================================================

# --- country-level Spearman ---

# We need all_theta_country_summary from cell 16.
# It should contain columns: [theta, country_name, Slack_weighted, n_critical_products]

country_spearman_rows = []

for t_a, t_b in [(a, b) for a in THETA_VALUES for b in THETA_VALUES if a <= b]:

    sub_a = all_theta_country_summary[
        all_theta_country_summary["theta"] == t_a
    ][["country_name", "Slack_weighted"]].rename(
        columns={"Slack_weighted": "sw_a"}
    )

    sub_b = all_theta_country_summary[
        all_theta_country_summary["theta"] == t_b
    ][["country_name", "Slack_weighted"]].rename(
        columns={"Slack_weighted": "sw_b"}
    )

    merged = sub_a.merge(sub_b, on="country_name", how="inner")

    if len(merged) < 5:
        continue

    rho, pval = spearmanr(merged["sw_a"], merged["sw_b"])

    country_spearman_rows.append({
        "theta_a": t_a,
        "theta_b": t_b,
        "n_countries": len(merged),
        "spearman_rho": round(rho, 4),
        "p_value": round(pval, 6),
        "set_type": "country_Slack_weighted"
    })

# --- product-level Spearman ---

product_spearman_rows = []

for t_a, t_b in [(a, b) for a in THETA_VALUES for b in THETA_VALUES if a <= b]:

    sub_a = all_theta_product_summary[
        all_theta_product_summary["theta"] == t_a
    ][["hs4", "product_fragility_score"]].rename(
        columns={"product_fragility_score": "pf_a"}
    )

    sub_b = all_theta_product_summary[
        all_theta_product_summary["theta"] == t_b
    ][["hs4", "product_fragility_score"]].rename(
        columns={"product_fragility_score": "pf_b"}
    )

    merged = sub_a.merge(sub_b, on="hs4", how="inner")

    if len(merged) < 5:
        continue

    rho, pval = spearmanr(merged["pf_a"], merged["pf_b"])

    product_spearman_rows.append({
        "theta_a": t_a,
        "theta_b": t_b,
        "n_products": len(merged),
        "spearman_rho": round(rho, 4),
        "p_value": round(pval, 6),
        "set_type": "product_fragility_score"
    })

spearman_results = pd.DataFrame(country_spearman_rows + product_spearman_rows)

print("\n========== SPEARMAN RANK CORRELATION ACROSS THETA VALUES ==========")
display(spearman_results)

spearman_path = os.path.join(ROBUST_TABLE_DIR, f"spearman_rank_correlation_{YEAR}.csv")
spearman_results.to_csv(spearman_path, index=False)
print("Saved:", spearman_path)

# --- Spearman heatmap ---

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, stype, label in zip(
    axes,
    ["country_Slack_weighted", "product_fragility_score"],
    ["Country: Slack_weighted", "Product: fragility score"]
):
    sub = spearman_results[spearman_results["set_type"] == stype].copy()

    mat = pd.DataFrame(index=THETA_VALUES, columns=THETA_VALUES, dtype=float)
    for _, row in sub.iterrows():
        mat.loc[row["theta_a"], row["theta_b"]] = row["spearman_rho"]
        mat.loc[row["theta_b"], row["theta_a"]] = row["spearman_rho"]

    mat = mat.astype(float)

    sns.heatmap(
        mat,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0.8,          # Spearman rho should be high throughout
        vmax=1.0,
        linewidths=0.5,
        linecolor="white",
        cbar=True,
        annot_kws={"size": 10}
    )

    baseline_idx = THETA_VALUES.index(BASELINE_THETA)
    for offset in [0, 1]:
        ax.axhline(y=baseline_idx + offset, color="steelblue", lw=2, ls="--")
        ax.axvline(x=baseline_idx + offset, color="steelblue", lw=2, ls="--")

    ax.set_title(f"Spearman ρ  —  {label}", fontsize=11, fontweight="bold")
    ax.set_xlabel("θ")
    ax.set_ylabel("θ")
    ax.set_xticklabels([str(t) for t in THETA_VALUES])
    ax.set_yticklabels([str(t) for t in THETA_VALUES], rotation=0)

fig.suptitle(
    f"Rank Stability of Slack-Based Measures Across θ Values  |  Year {YEAR}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
spearman_fig_path = os.path.join(ROBUST_FIG_DIR, f"spearman_heatmap_{YEAR}.png")
plt.savefig(spearman_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", spearman_fig_path)


# ============================================================
# C. STABLE-CORE TABLES
#
# Identify countries / products / Italy import routes that are
# critical under ALL five theta values.
# These are the most defensible findings for the paper.
# ============================================================

print("\n\n========== STABLE CORE ANALYSIS ==========")
print("Critical under ALL theta values in", THETA_VALUES)

# --- C1. Stable-core countries (globally critical under all theta) ---

all_critical_country_sets = {}

for t in THETA_VALUES:
    sub = all_theta_country_summary[
        (all_theta_country_summary["theta"] == t) &
        (all_theta_country_summary["n_critical_products"] > 0)
    ]
    all_critical_country_sets[t] = set(sub["country_name"].unique())

stable_core_countries = set.intersection(*all_critical_country_sets.values())

stable_core_country_df = (
    all_theta_country_summary[
        (all_theta_country_summary["theta"] == BASELINE_THETA) &
        (all_theta_country_summary["country_name"].isin(stable_core_countries))
    ]
    .sort_values("n_critical_products", ascending=False)
    [[
        "country_name",
        "n_critical_products",
        "n_severe_products",
        "Slack_weighted",
        "critical_export_value",
        "critical_export_share",
        "max_supply_share"
    ]]
    .copy()
)

stable_core_country_df["n_theta_values"] = len(THETA_VALUES)

print(f"\nStable-core COUNTRIES (critical under all {len(THETA_VALUES)} theta values):",
      len(stable_core_countries))
display(stable_core_country_df.head(25))

stable_core_country_path = os.path.join(
    ROBUST_TABLE_DIR, f"stable_core_countries_{YEAR}.csv"
)
stable_core_country_df.to_csv(stable_core_country_path, index=False)
print("Saved:", stable_core_country_path)


# --- C2. Stable-core products (globally critical under all theta) ---

all_critical_product_sets = {}

for t in THETA_VALUES:
    sub = all_theta_product_summary[
        (all_theta_product_summary["theta"] == t) &
        (all_theta_product_summary["n_critical_suppliers"] > 0)
    ]
    all_critical_product_sets[t] = set(sub["hs4"].unique())

stable_core_products = set.intersection(*all_critical_product_sets.values())

stable_core_product_df = (
    all_theta_product_summary[
        (all_theta_product_summary["theta"] == BASELINE_THETA) &
        (all_theta_product_summary["hs4"].isin(stable_core_products))
    ]
    .sort_values("product_fragility_score", ascending=False)
    [[
        "hs4",
        "hs2",
        "product_description",
        "total_trade",
        "n_exporters",
        "effective_n_exporters",
        "dominant_supplier",
        "dominant_supplier_share",
        "HHI",
        "n_critical_suppliers",
        "product_fragility_score",
        "systemic_fragility_rank"
        if "systemic_fragility_rank" in all_theta_product_summary.columns
        else "product_fragility_score"
    ]]
    .drop_duplicates("hs4")
    .copy()
)

stable_core_product_df["n_theta_values"] = len(THETA_VALUES)

print(f"\nStable-core PRODUCTS (critical under all {len(THETA_VALUES)} theta values):",
      len(stable_core_products))
display(stable_core_product_df.head(25))

stable_core_product_path = os.path.join(
    ROBUST_TABLE_DIR, f"stable_core_products_{YEAR}.csv"
)
stable_core_product_df.to_csv(stable_core_product_path, index=False)
print("Saved:", stable_core_product_path)


# --- C3. Stable-core Italy import routes (critical under all theta) ---

# We need all_theta_italy_routes from cell 16.
# It should contain columns: [theta, supplier_country, hs4, Slack_import_cp]

# Check if all_theta_italy_routes was created (i.e., if Italy robustness was enabled)
if not all_theta_italy_routes.empty:

    all_critical_italy_sets = {}

    for t in THETA_VALUES:
        sub = all_theta_italy_routes[
            (all_theta_italy_routes["theta"] == t) &
            (all_theta_italy_routes["Slack_import_cp"] < 1)
        ]
        route_keys = sub["supplier_country"] + "||" + sub["hs4"].astype(str)
        all_critical_italy_sets[t] = set(route_keys)

    stable_core_italy_routes = set.intersection(*all_critical_italy_sets.values())

    baseline_italy = all_theta_italy_routes[
        all_theta_italy_routes["theta"] == BASELINE_THETA
    ].copy()

    baseline_italy["route_key"] = (
        baseline_italy["supplier_country"] + "||" + baseline_italy["hs4"].astype(str)
    )

    stable_core_italy_df = (
        baseline_italy[
            baseline_italy["route_key"].isin(stable_core_italy_routes)
        ]
        .sort_values("Slack_import_cp", ascending=True)
        [[
            "supplier_country",
            "hs4",
            "hs2",
            "product_description",
            "weight",
            "supplier_share",
            "Slack_import_cp",
            "risk_label"
        ]]
        .copy()
    )

    stable_core_italy_df["n_theta_values"] = len(THETA_VALUES)

    print(f"\nStable-core ITALY IMPORT ROUTES (critical under all {len(THETA_VALUES)} theta values):",
          len(stable_core_italy_routes))
    display(stable_core_italy_df.head(30))

    stable_core_italy_path = os.path.join(
        ROBUST_TABLE_DIR, f"stable_core_italy_import_routes_{YEAR}.csv"
    )
    stable_core_italy_df.to_csv(stable_core_italy_path, index=False)
    print("Saved:", stable_core_italy_path)

else:
    print("\nall_theta_italy_routes not found or empty. Re-run cell 16 with Italy robustness enabled.")


# ============================================================
# D. SUMMARY TABLE FOR PAPER
# One compact table with all Jaccard means and the
# adjacent-theta (±0.05) comparisons from baseline.
# ============================================================

summary_rows = []

for stype in set_types:
    sub = jaccard_results[
        (jaccard_results["set_type"] == stype) &
        (jaccard_results["theta_a"] == BASELINE_THETA)
    ].copy()

    row = {"set_type": stype}

    for _, r in sub.iterrows():
        row[f"jaccard_vs_{r['theta_b']}"] = round(r["jaccard"], 3)

    # Mean across all non-self comparisons
    non_self = jaccard_results[
        (jaccard_results["set_type"] == stype) &
        (jaccard_results["theta_a"] != jaccard_results["theta_b"])
    ]["jaccard"]

    row["mean_jaccard"] = round(non_self.mean(), 3)
    row["min_jaccard"]  = round(non_self.min(), 3)

    summary_rows.append(row)

robustness_summary = pd.DataFrame(summary_rows)

print("\n========== ROBUSTNESS SUMMARY TABLE (for paper) ==========")
display(robustness_summary)

robustness_summary_path = os.path.join(
    ROBUST_TABLE_DIR, f"robustness_summary_table_{YEAR}.csv"
)
robustness_summary.to_csv(robustness_summary_path, index=False)
print("Saved:", robustness_summary_path)

In [ ]:
import os

print(f"Output Directory: {OUT_DIR}\n")

print("--- Saved CSV Tables ---")
for var_name, filename in tables_to_save.items():
    print(os.path.join(TABLE_DIR, filename))

print("\n--- Saved Excel Workbook ---")
print(excel_path)

print("\n--- Saved Figures (PNG and HTML) ---")

# General figures saved in cell 15
print(os.path.join(FIG_DIR, f"distribution_country_weighted_slack_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"distribution_product_fragility_score_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"top_pivotal_countries_weighted_slack_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"top_fragile_products_fragility_score_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"distribution_top_supplier_share_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"hhi_vs_product_fragility_robustness_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"systemic_fragile_products_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"italy_critical_export_products_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"italy_top_critical_import_routes_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"italy_supplier_country_exposure_{YEAR}.png"))

# Figures from cell 14G
print(os.path.join(FIG_DIR, f"italy_critical_import_supplier_map_{YEAR}.html"))
print(os.path.join(FIG_DIR, f"italy_critical_import_supplier_map_{YEAR}.png"))
print(os.path.join(FIG_DIR, f"italy_top_critical_import_suppliers_{YEAR}.html"))
print(os.path.join(FIG_DIR, f"italy_top_critical_import_suppliers_{YEAR}.png"))

# Figures from cell 14I (World map of Italy's critical import routes)
print(os.path.join(FIG_DIR, f"italy_critical_import_routes_map_{YEAR}.html"))
print(os.path.join(FIG_DIR, f"italy_critical_import_routes_map_{YEAR}.png"))

# Figures from cell 14J (Product-specific critical route map)
# Assuming the example HS4 2002 was used
print(os.path.join(FIG_DIR, f"italy_hs4_2002_supplier_routes_{YEAR}.html"))
print(os.path.join(FIG_DIR, f"italy_hs4_2002_supplier_routes_{YEAR}.png"))

# Figures from cell 14K (Country-node network graph)
print(os.path.join(FIG_DIR, f"italy_country_node_critical_import_network_{YEAR}.png"))

# Figures from cell 14L (Sankey Diagram)
print(os.path.join(FIG_DIR, f"italy_critical_import_routes_sankey_{YEAR}.html"))
print(os.path.join(FIG_DIR, f"italy_critical_import_routes_sankey_{YEAR}.png"))

# Robustness figures (from cell 16)
print(os.path.join(ROBUST_FIG_DIR, f"italy_import_critical_routes_by_theta_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"global_critical_edges_by_theta_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"critical_links_by_min_weight_{YEAR}.png"))
for set_type in set_types:
    print(os.path.join(ROBUST_FIG_DIR, f"jaccard_{set_type}_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"hhi_vs_product_fragility_robustness_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"dominant_supplier_share_vs_product_fragility_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"jaccard_heatmaps_all_sets_{YEAR}.png"))
print(os.path.join(ROBUST_FIG_DIR, f"spearman_heatmap_{YEAR}.png"))


print("\n--- Metadata and Pickle Files ---")
print(metadata_path)
print(pickle_path)

Output Directory: /content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack

--- Saved CSV Tables ---
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/edges_country_product_slack_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/country_weighted_slack_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/product_fragility_summary_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/critical_country_product_edges_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/criticality_contribution_edges_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/top_pivotal_countries_by_weighted_slack_2023.csv
/content/drive/MyDrive/BACI_HS17_V202601_/results_country_product_weighted_slack/tables/top_fragile_products_2023.